# 🎙️ Matraca Studio - Dublador & Clonador de Voz com IA
**Clone sua voz e duble qualquer áudio para múltiplos idiomas com sincronia temporal e suporte a legendas SRT!**

Desenvolvido para execução otimizada no Google Colab com GPU (T4, L4, A100).
- **Motores de Clonagem Vocal (1 modelo ativo por vez na GPU):**
  - ⚡ **Qwen3-TTS 1.7B Base** ([QwenLM/Qwen3-TTS](https://github.com/QwenLM/Qwen3-TTS)) — Modelo autorregressivo de 1.7B parâmetros da Alibaba Cloud com altíssima expressividade (9 idiomas suportados).
  - 🎙️ **OmniVoice** ([k2-fsa/OmniVoice](https://github.com/k2-fsa/OmniVoice)) — Modelo de difusão acústica da k2-fsa (suporta os 10 idiomas, incluindo Árabe).
- **IA de Reconhecimento de Fala & Timestamps:** OpenAI Whisper
- **Processamento de Áudio:** FFmpeg + Torchaudio (Time-stretching suave preservando timbre original)
- **Sincronização por Legendas:** Importação de arquivos `.SRT` com síntese temporalmente alinhada
- **Resiliência e Anti-Timeout:** Sentinela anti-inatividade no Google Colab e heartbeat no navegador com áudio inaudível, impedindo suspensão de abas e timeout do runtime.
- **Gerenciador de Jobs Persistente:** Execução desacoplada em threads de segundo plano. Se a página sofrer recarregamento (F5) ou oscilação de rede, a síntese continua ativa e a interface recupera automaticamente o status, logs e downloads.
- **Interface:** Gradio com fila sequencial e limpeza contínua de VRAM

In [ ]:
# @title Passo 1: Instalar Dependências e FFmpeg
# @markdown Instala Qwen3-TTS, OmniVoice, Whisper, Gradio, Deep-Translator e ferramentas de mídia.

!apt-get -y update -qq && apt-get -y install -qq ffmpeg sox libsox-fmt-all
!pip install -q sox onnxruntime gradio openai-whisper deep-translator tensorboardx webdataset soxr
!pip install -q transformers==4.57.3
!pip install -q --no-deps qwen-tts omnivoice

# Baixa a ponte de compatibilidade acústica para o OmniVoice rodar em perfeita harmonia com o Qwen3-TTS
import os
import urllib.request

if not os.path.exists("higgs_audio_v2_tokenizer.py"):
    url = "https://raw.githubusercontent.com/dgurgel-info/ai-colab-matracastudio/main/higgs_audio_v2_tokenizer.py"
    try:
        urllib.request.urlretrieve(url, "higgs_audio_v2_tokenizer.py")
    except Exception as e:
        print(f"Aviso ao baixar higgs_audio_v2_tokenizer.py: {e}")

print('✅ Dependências e ferramentas de mídia instaladas com sucesso!')

In [ ]:
# @title Passo 2: Carregar os Modelos de IA na GPU (T4 / A100 / L4)
# @markdown Baixa e carrega o Whisper e inicializa 1 modelo de TTS por vez na memória de vídeo (VRAM).

import os
import gc
import logging
import warnings
import torch
import torchaudio
import whisper

# Suprime avisos redundantes do gerador transformers
logging.getLogger('transformers.generation.utils').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', message='.*Setting `pad_token_id` to `eos_token_id`.*')

# Ponte de compatibilidade acústica entre OmniVoice e Qwen3-TTS
import transformers
from transformers.tokenization_utils_base import PreTrainedTokenizerBase

# Suporte a listas e dicionários de tokens especiais no transformers 4.57.3 (evita AttributeError / RecursionError)
def _safe_set_model_specific_special_tokens(self, special_tokens):
    if not special_tokens:
        return
    if isinstance(special_tokens, dict):
        self.SPECIAL_TOKENS_ATTRIBUTES = list(set(self.SPECIAL_TOKENS_ATTRIBUTES + list(special_tokens.keys())))
        for key, value in special_tokens.items():
            self._special_tokens_map[key] = value
    elif isinstance(special_tokens, (list, tuple, set)):
        self.SPECIAL_TOKENS_ATTRIBUTES = list(set(self.SPECIAL_TOKENS_ATTRIBUTES + list(special_tokens)))
        for key in special_tokens:
            self._special_tokens_map[str(key)] = str(key)

PreTrainedTokenizerBase._set_model_specific_special_tokens = _safe_set_model_specific_special_tokens

from higgs_audio_v2_tokenizer import HiggsAudioV2TokenizerConfig, HiggsAudioV2TokenizerModel
if hasattr(transformers, '_objects'):
    transformers._objects['HiggsAudioV2TokenizerModel'] = HiggsAudioV2TokenizerModel
    transformers._objects['HiggsAudioV2TokenizerConfig'] = HiggsAudioV2TokenizerConfig
transformers.HiggsAudioV2TokenizerModel = HiggsAudioV2TokenizerModel
transformers.HiggsAudioV2TokenizerConfig = HiggsAudioV2TokenizerConfig
from transformers.models.auto import CONFIG_MAPPING
CONFIG_MAPPING['higgs_audio_v2_tokenizer'] = HiggsAudioV2TokenizerConfig

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32

print(f'⚙️ Dispositivo em uso: {device}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'🖥️ GPU Detectada: {gpu_name} ({vram:.1f} GB VRAM)')
else:
    print('⚠️ GPU não detectada! Por favor, ative a GPU T4 no menu do Colab (Ambiente de Execução > Alterar tipo de ambiente de execução).')

# Atualizado para o modelo 'small' (244M parâmetros): precisão significativamente superior
# na detecção de Português Brasileiro (pt-BR), evitando alucinações e erros de detecção em inglês.
print('\n📝 Carregando modelo Whisper (small) para transcrição de áudio...')
whisper_model = whisper.load_model('small', device=device)
print('✅ Whisper pronto!')

# Gestão estrita de VRAM: Carrega exatamente 1 modelo de TTS por vez.
# Ao trocar de modelo, o anterior é completamente descarregado da GPU.
current_tts_model_name = None
active_tts_model = None

def load_tts_model(model_choice):
    """
    Garante que apenas UM modelo de TTS resida na memória da GPU por vez.
    Se o usuário trocar de modelo, descarrega o modelo anterior e libera a VRAM antes de carregar o novo.
    """
    global current_tts_model_name, active_tts_model

    is_qwen = 'Qwen' in str(model_choice)
    target_name = 'Qwen3-TTS' if is_qwen else 'OmniVoice'

    # Se o modelo desejado já estiver ativo, retorna imediatamente
    if current_tts_model_name == target_name and active_tts_model is not None:
        return target_name, active_tts_model

    # Se outro modelo estiver ativo na GPU, descarrega-o completamente
    if active_tts_model is not None:
        print(f'\n🧹 Descarregando [{current_tts_model_name}] da memória da GPU...')
        del active_tts_model
        active_tts_model = None
        current_tts_model_name = None
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        gc.collect()
        print('✅ VRAM liberada!')

    # Carrega exclusivamente o modelo escolhido
    if target_name == 'Qwen3-TTS':
        print('\n🤖 Carregando Qwen3-TTS 1.7B Base (Alibaba Cloud) na GPU (float16)...')
        from qwen_tts import Qwen3TTSModel
        active_tts_model = Qwen3TTSModel.from_pretrained(
            'Qwen/Qwen3-TTS-12Hz-1.7B-Base',
            device_map=device,
            dtype=dtype
        )
        # Define pad_token_id para eliminar silenciosamente o aviso do transformers
        if hasattr(active_tts_model.model, 'generation_config'):
            active_tts_model.model.generation_config.pad_token_id = active_tts_model.model.generation_config.eos_token_id
        if hasattr(active_tts_model, 'generate_defaults') and hasattr(active_tts_model.generate_defaults, 'pad_token_id'):
            active_tts_model.generate_defaults.pad_token_id = active_tts_model.model.generation_config.eos_token_id
        current_tts_model_name = 'Qwen3-TTS'
        print('✅ Qwen3-TTS 1.7B carregado com sucesso na GPU!')
    else:
        print('\n🎙️ Carregando OmniVoice (k2-fsa) na GPU...')
        from omnivoice import OmniVoice
        active_tts_model = OmniVoice.from_pretrained(
            'k2-fsa/OmniVoice',
            device_map=device,
            dtype=dtype
        )
        current_tts_model_name = 'OmniVoice'
        print('✅ OmniVoice carregado com sucesso na GPU!')

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return target_name, active_tts_model

# Carrega o modelo padrão inicial (Qwen3-TTS 1.7B)
print('\n⚡ Carregando motor padrão inicial...')
load_tts_model('Qwen3-TTS')
print('\n🚀 Ambiente preparado e pronto para uso!')


In [ ]:
# @title Passo 3: Motor de Processamento, Sincronização Temporal, Legendas SRT e Síntese
# @markdown Funções auxiliares para extração de áudio, medição de duração, normalização, time stretch suave, parser de SRT, síntese sequencial streaming, cancelamento (STOP) e limpeza contínua de cache.

import subprocess
import json
import tempfile
import os
import re
import shutil
import gc
import time
import torch
import torchaudio
import numpy as np
from deep_translator import GoogleTranslator, MyMemoryTranslator

# Injeção global de User-Agent de navegador para requisições HTTP (evita bloqueio do Google Translate e erro 500)
import requests
_orig_requests_get = requests.get
def _browser_requests_get(url, *args, **kwargs):
    headers = kwargs.get('headers')
    if headers is None:
        headers = {}
    else:
        headers = dict(headers)
    if 'User-Agent' not in headers:
        headers['User-Agent'] = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
    kwargs['headers'] = headers
    return _orig_requests_get(url, *args, **kwargs)
requests.get = _browser_requests_get

# Flag global para controle do botão STOP e cancelamento de geração imediato
STOP_REQUESTED = False

def set_stop_requested(val=True):
    global STOP_REQUESTED
    STOP_REQUESTED = val

def is_stop_requested():
    global STOP_REQUESTED
    return STOP_REQUESTED

def get_audio_duration(file_path):
    """Retorna a duração em segundos do arquivo de áudio usando ffprobe."""
    if not file_path or not os.path.exists(file_path):
        return 0.0
    cmd = [
        'ffprobe', '-v', 'error',
        '-show_entries', 'format=duration',
        '-of', 'json', file_path
    ]
    try:
        res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        data = json.loads(res.stdout)
        return float(data.get('format', {}).get('duration', 0.0))
    except Exception as e:
        print(f'Erro ao inspecionar áudio com ffprobe: {e}')
        return 0.0

def extract_audio_to_wav(media_path, output_wav):
    """Converte qualquer áudio (WAV, MP3, M4A, OGG, FLAC) em WAV 24kHz mono puro para clonagem de voz."""
    cmd = [
        'ffmpeg', '-y', '-i', media_path,
        '-vn', '-acodec', 'pcm_s16le', '-ar', '24000', '-ac', '1',
        output_wav
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

def extract_audio_slice(input_wav, start_sec, end_sec, output_slice_wav):
    """Extrai uma fatia pura de áudio com recorte exato."""
    duration = max(0.5, end_sec - start_sec)
    cmd = [
        'ffmpeg', '-y',
        '-ss', f'{start_sec:.3f}',
        '-t', f'{duration:.3f}',
        '-i', input_wav,
        '-acodec', 'pcm_s16le', '-ar', '24000', '-ac', '1',
        output_slice_wav
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

def normalize_audio_tensor(tensor, target_peak=0.95):
    """Normaliza o volume do tensor de áudio para evitar clipping ou volume muito baixo."""
    if tensor is None or tensor.numel() == 0:
        return tensor
    max_val = torch.max(torch.abs(tensor))
    if max_val > 1e-4:
        return tensor / max_val * target_peak
    return tensor

def select_best_voice_slice(segments, full_wav, orig_duration, output_slice_wav, whisper_model=None, orig_lang='pt'):
    """
    Identifica o trecho vocal mais limpo e representativo (4 a 8s),
    evitando músicas, vinhetas ou ruídos.
    Gera também o texto 100% alinhado com a fatia para a clonagem.
    """
    ref_slice_start = 0.0
    ref_slice_end = min(orig_duration, 6.0)

    valid_segments = []
    if segments:
        for s in segments:
            txt = s.get('text', '').strip()
            if not txt or re.search(r'(\[música\]|\[music\]|♪|♫|\(música\))', txt, re.IGNORECASE):
                continue
            if len(txt) >= 8 and (s['end'] - s['start']) >= 1.0:
                valid_segments.append(s)

    if valid_segments:
        best_combo = []
        for i in range(len(valid_segments)):
            combo = [valid_segments[i]]
            dur = combo[-1]['end'] - combo[0]['start']
            for j in range(i + 1, len(valid_segments)):
                if valid_segments[j]['start'] - valid_segments[j-1]['end'] < 1.5:
                    combo.append(valid_segments[j])
                    dur = combo[-1]['end'] - combo[0]['start']
                    if dur >= 5.0:
                        break
                else:
                    break
            if 4.0 <= dur <= 10.0:
                best_combo = combo
                break
            elif dur > (best_combo[-1]['end'] - best_combo[0]['start'] if best_combo else 0):
                best_combo = combo

        if best_combo:
            ref_slice_start = max(0.0, best_combo[0]['start'])
            ref_slice_end = min(orig_duration, best_combo[-1]['end'])

    extract_audio_slice(full_wav, ref_slice_start, ref_slice_end, output_slice_wav)

    ref_text = ""
    if whisper_model is not None:
        try:
            lang_arg = None if (not orig_lang or orig_lang == 'auto') else orig_lang
            asr = whisper_model.transcribe(output_slice_wav, language=lang_arg, task='transcribe')
            ref_text = asr.get('text', '').strip()
        except Exception:
            pass

    return ref_slice_start, ref_slice_end, ref_text

def split_text_into_chunks(text, max_chunk_chars=260):
    """
    Divide textos longos em blocos naturais e respiráveis.
    Respeita pontuação principal (. ! ?), secundária (, ; : -) ou quebra por palavras
    garantindo que nenhum bloco ultrapasse max_chunk_chars.
    """
    if not text or not text.strip():
        return []

    sentences = re.split(r'([.!?;:\n]+)', text.strip())
    raw_parts = []
    i = 0
    while i < len(sentences):
        s = sentences[i].strip()
        punct = sentences[i+1].strip() if i+1 < len(sentences) else ''
        full = f"{s}{punct}".strip()
        if full:
            raw_parts.append(full)
        i += 2

    refined_parts = []
    for p in raw_parts:
        if len(p) <= max_chunk_chars:
            refined_parts.append(p)
        else:
            sub_clauses = re.split(r'([,–—\-])', p)
            temp_clause = []
            j = 0
            while j < len(sub_clauses):
                c = sub_clauses[j].strip()
                cpunct = sub_clauses[j+1].strip() if j+1 < len(sub_clauses) else ''
                cfull = f"{c}{cpunct}".strip()
                if cfull:
                    temp_clause.append(cfull)
                j += 2

            curr_sub = ""
            for tc in temp_clause:
                if len(curr_sub) + len(tc) + 1 <= max_chunk_chars:
                    curr_sub = f"{curr_sub} {tc}".strip()
                else:
                    if curr_sub:
                        refined_parts.append(curr_sub)
                    if len(tc) <= max_chunk_chars:
                        curr_sub = tc
                    else:
                        words = tc.split()
                        w_chunk = ""
                        for w in words:
                            if len(w_chunk) + len(w) + 1 <= max_chunk_chars:
                                w_chunk = f"{w_chunk} {w}".strip()
                            else:
                                if w_chunk:
                                    refined_parts.append(w_chunk)
                                w_chunk = w
                        curr_sub = w_chunk
            if curr_sub:
                refined_parts.append(curr_sub)

    final_chunks = []
    current = ""
    for part in refined_parts:
        if not current:
            current = part
        elif len(current) + len(part) + 1 <= max_chunk_chars:
            current = f"{current} {part}".strip()
        else:
            final_chunks.append(current)
            current = part
    if current:
        final_chunks.append(current)

    return [c for c in final_chunks if c.strip()]

class TranslationUnavailableError(RuntimeError):
    """Impede a síntese quando nenhum provedor entrega uma tradução válida."""

def _normalize_translation_code(language_code):
    code = (language_code or 'auto').lower().replace('_', '-')
    if code == 'pt-br':
        return 'pt'
    if code == 'zh-cn':
        return 'zh-CN'
    return code

def _language_family(language_code):
    return _normalize_translation_code(language_code).lower().split('-')[0]

def is_translation_valid(translated_str, original_chunk=None, source_code='auto', target_code=None):
    """Rejeita respostas vazias, HTML, erros de provedor e texto não traduzido."""
    if not translated_str or not isinstance(translated_str, str):
        return False
    t_clean = translated_str.strip()
    if not t_clean:
        return False
    t_lower = t_clean.lower()
    error_signatures = [
        'error 500', 'server error', "that’s an error", "that's an error",
        'there was an error', 'please try again later', '<html', '<div',
        'af-error-container', 'af-error-page', 'result-container', 'quota exceeded'
    ]
    if any(sig in t_lower for sig in error_signatures):
        return False

    # Se a origem e o destino forem idiomas diferentes, uma resposta idêntica
    # não é tradução e nunca deve ser enviada ao motor de voz.
    if original_chunk and target_code and _language_family(source_code) != _language_family(target_code):
        if t_clean.casefold() == original_chunk.strip().casefold():
            return False
    return True

def translate_text_robust(text, target_code, source_code='auto', max_chunk=700, progress_callback=None):
    """
    Traduz com Google como primeira opção e MyMemory como fallback gratuito.
    Se os dois serviços falharem, lança erro antes da síntese: o texto original
    jamais é usado como substituto silencioso para um idioma de destino.
    """
    if not text or not text.strip():
        return ''

    clean_text = text.strip()
    api_target = _normalize_translation_code(target_code)
    api_source = _normalize_translation_code(source_code)

    # Não chama serviços externos quando a dublagem permanece no mesmo idioma.
    if api_source != 'auto' and _language_family(api_source) == _language_family(api_target):
        return clean_text

    chunks = split_text_into_chunks(clean_text, max_chunk_chars=max_chunk)
    if not chunks:
        return ''

    providers = []
    for provider_name, provider_class, retries in (
        ('Google Tradutor', GoogleTranslator, 2),
        ('MyMemory', MyMemoryTranslator, 2),
    ):
        try:
            providers.append((provider_name, provider_class(source=api_source, target=api_target), retries))
        except Exception as error:
            message = f'⚠️ {provider_name} indisponível para {api_target}: {type(error).__name__}.'
            print(message)
            if progress_callback:
                progress_callback(message)

    if not providers:
        raise TranslationUnavailableError('Nenhum provedor de tradução pôde ser inicializado. Nenhum áudio foi gerado.')

    translated_parts = []
    total_chunks = len(chunks)
    for idx, chunk in enumerate(chunks):
        if is_stop_requested():
            raise InterruptedError('Operação cancelada pelo usuário.')

        translated_part = None
        failures = []
        for provider_name, translator, retries in providers:
            for attempt in range(retries):
                try:
                    result = translator.translate(chunk)
                    if is_translation_valid(result, chunk, source_code=api_source, target_code=api_target):
                        translated_part = result.strip()
                        if provider_name != 'Google Tradutor':
                            message = f'ℹ️ Bloco [{idx+1}/{total_chunks}] traduzido com MyMemory após indisponibilidade do Google.'
                            print(message)
                            if progress_callback:
                                progress_callback(message)
                        break
                    failures.append(f'{provider_name}: resposta inválida')
                except Exception as error:
                    failures.append(f'{provider_name}: {type(error).__name__}')
                time.sleep(0.4 * (attempt + 1))
            if translated_part is not None:
                break

        if translated_part is None:
            detail = '; '.join(failures[-4:]) or 'sem resposta válida'
            message = (
                f'❌ Não foi possível traduzir o bloco [{idx+1}/{total_chunks}] para {api_target} '
                f'({detail}). Nenhum áudio será gerado para este idioma.'
            )
            print(f'\n{message}')
            if progress_callback:
                progress_callback(message)
            raise TranslationUnavailableError(message)

        translated_parts.append(translated_part)

    return ' '.join(translated_parts)

def build_atempo_filter(speed_factor):
    """Gera encadeamento de filtros atempo no FFmpeg (cada filtro suporta entre 0.5 e 2.0)."""
    speed = speed_factor
    filters = []
    while speed > 2.0:
        filters.append('atempo=2.0')
        speed /= 2.0
    while speed < 0.5:
        filters.append('atempo=0.5')
        speed /= 0.5
    filters.append(f'atempo={speed:.5f}')
    return ','.join(filters)

def time_sync_audio(synth_wav_path, target_duration, output_synced_wav, max_stretch=0.18):
    """
    Ajusta a duração da fala gerada preservando a qualidade acústica natural.
    Limita o time-stretch a no máximo ±18% para evitar vozes robóticas.
    """
    synth_duration = get_audio_duration(synth_wav_path)
    if synth_duration <= 0 or target_duration <= 0:
        shutil.copyfile(synth_wav_path, output_synced_wav)
        return 1.0, synth_duration

    raw_speed_factor = synth_duration / target_duration
    speed_factor = max(1.0 - max_stretch, min(1.0 + max_stretch, raw_speed_factor))

    if 0.98 <= speed_factor <= 1.02:
        filter_chain = f'apad=whole_dur={target_duration:.4f}'
    else:
        tempo_filter = build_atempo_filter(speed_factor)
        filter_chain = f'{tempo_filter},apad=whole_dur={target_duration:.4f}'

    cmd = [
        'ffmpeg', '-y', '-i', synth_wav_path,
        '-filter:a', filter_chain,
        '-t', f'{target_duration:.4f}',
        '-acodec', 'pcm_s16le', '-ar', '24000', '-ac', '1',
        output_synced_wav
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    return raw_speed_factor, target_duration

def assemble_timeline_audio(
    original_audio_path,
    synth_speech_wav_path,
    speech_start,
    speech_end,
    total_duration,
    output_final_wav,
    sync_duration=True
):
    """Monta a linha do tempo inteligente preservando músicas de introdução e encerramento."""
    synth_dur = get_audio_duration(synth_speech_wav_path)
    target_speech_dur = max(0.5, speech_end - speech_start)
    has_intro = speech_start >= 0.25
    has_outro = (total_duration - speech_end) >= 0.25

    temp_synced_speech = tempfile.NamedTemporaryFile(suffix='_synced_speech.wav', delete=False).name
    if sync_duration and target_speech_dur > 0:
        raw_speed, _ = time_sync_audio(synth_speech_wav_path, target_speech_dur, temp_synced_speech)
    else:
        shutil.copyfile(synth_speech_wav_path, temp_synced_speech)
        raw_speed = 1.0

    audio_parts = []

    if has_intro:
        temp_intro = tempfile.NamedTemporaryFile(suffix='_intro.wav', delete=False).name
        cmd_intro = [
            'ffmpeg', '-y',
            '-ss', '0.000',
            '-t', f'{speech_start:.3f}',
            '-i', original_audio_path,
            '-acodec', 'pcm_s16le', '-ar', '24000', '-ac', '1',
            temp_intro
        ]
        subprocess.run(cmd_intro, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        t_intro, _ = torchaudio.load(temp_intro)
        audio_parts.append(t_intro)

    t_speech, _ = torchaudio.load(temp_synced_speech)
    t_speech = normalize_audio_tensor(t_speech, target_peak=0.95)
    audio_parts.append(t_speech)

    if has_outro and sync_duration:
        outro_dur = max(0.1, total_duration - speech_end)
        temp_outro = tempfile.NamedTemporaryFile(suffix='_outro.wav', delete=False).name
        cmd_outro = [
            'ffmpeg', '-y',
            '-ss', f'{speech_end:.3f}',
            '-t', f'{outro_dur:.3f}',
            '-i', original_audio_path,
            '-acodec', 'pcm_s16le', '-ar', '24000', '-ac', '1',
            temp_outro
        ]
        subprocess.run(cmd_outro, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        t_outro, _ = torchaudio.load(temp_outro)
        audio_parts.append(t_outro)

    t_final = torch.cat(audio_parts, dim=-1)
    temp_cat = tempfile.NamedTemporaryFile(suffix='_cat.wav', delete=False).name
    torchaudio.save(temp_cat, t_final, 24000)

    if sync_duration and total_duration > 0:
        cmd_fit = [
            'ffmpeg', '-y', '-i', temp_cat,
            '-filter:a', f'apad=whole_dur={total_duration:.4f}',
            '-t', f'{total_duration:.4f}',
            '-acodec', 'pcm_s16le', '-ar', '24000', '-ac', '1',
            output_final_wav
        ]
        subprocess.run(cmd_fit, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    else:
        shutil.copyfile(temp_cat, output_final_wav)

    final_dur = get_audio_duration(output_final_wav)
    return raw_speed, final_dur

def parse_srt_time(time_str):
    """Converte timestamp HH:MM:SS,mmm em segundos float."""
    time_str = time_str.strip().replace(',', '.')
    parts = time_str.split(':')
    if len(parts) == 3:
        return float(parts[0]) * 3600.0 + float(parts[1]) * 60.0 + float(parts[2])
    elif len(parts) == 2:
        return float(parts[0]) * 60.0 + float(parts[1])
    return float(parts[0])

def parse_srt(srt_file_path):
    """Lê um arquivo .srt e retorna lista de itens ordenados por tempo."""
    if not srt_file_path or not os.path.exists(srt_file_path):
        return []
    with open(srt_file_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()

    blocks = re.split(r'\n\s*\n', content.strip())
    entries = []
    for block in blocks:
        lines = [l.strip() for l in block.split('\n') if l.strip()]
        if len(lines) < 2:
            continue
        time_idx = -1
        for idx, l in enumerate(lines):
            if '-->' in l:
                time_idx = idx
                break
        if time_idx == -1:
            continue

        raw_text = ' '.join(lines[time_idx + 1:]).strip()
        clean_text = re.sub(r'<[^>]+>', '', raw_text).strip()
        time_tokens = lines[time_idx].split('-->')
        if len(time_tokens) == 2 and clean_text:
            try:
                start_s = parse_srt_time(time_tokens[0])
                end_s = parse_srt_time(time_tokens[1])
                if end_s > start_s:
                    entries.append({
                        'start': start_s,
                        'end': end_s,
                        'duration': end_s - start_s,
                        'text': clean_text
                    })
            except Exception:
                continue
    entries.sort(key=lambda x: x['start'])
    return entries

def generate_omnivoice_chunked_stream(model, text, ref_audio, ref_text, num_steps=32, speed=1.0):
    """Gerador streaming para síntese com OmniVoice."""
    chunks = split_text_into_chunks(text, max_chunk_chars=280)
    tensors = []
    silence = torch.zeros((1, int(24000 * 0.08)))

    header_msg = f"  🧩 [OmniVoice] Texto dividido em {len(chunks)} blocos para síntese:"
    print(f"\n{header_msg}")
    yield "header", header_msg, None

    for idx, c in enumerate(chunks):
        if is_stop_requested():
            print("\n⏹️ Interrupção solicitada pelo usuário!")
            raise InterruptedError("Operação cancelada pelo usuário.")

        preview = (c[:40] + '...') if len(c) > 40 else c
        start_line = f"    ▶️ Bloco [{idx+1}/{len(chunks)}] ({len(c)} chars): \"{preview}\" "
        print(start_line, end="", flush=True)
        yield "chunk_start", start_line, None

        t0 = time.time()
        out = model.generate(
            text=c,
            ref_audio=ref_audio,
            ref_text=ref_text if ref_text else None,
            num_step=int(num_steps),
            speed=float(speed)
        )
        t = out[0] if isinstance(out[0], torch.Tensor) else torch.tensor(out[0])
        if t.dim() == 1:
            t = t.unsqueeze(0)
        t = normalize_audio_tensor(t)
        tensors.append(t)
        tensors.append(silence)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        elapsed = time.time() - t0
        done_line = f"{start_line}✅ ({elapsed:.1f}s)"
        print(f"✅ ({elapsed:.1f}s)")
        yield "chunk_done", done_line, t

    if tensors:
        tensors.pop()
        combined = torch.cat(tensors, dim=-1)
        yield "complete", None, normalize_audio_tensor(combined)
    else:
        yield "complete", None, torch.zeros((1, 24000))

def generate_omnivoice_chunked(model, text, ref_audio, ref_text, num_steps=32, speed=1.0, progress_callback=None):
    """Gera fala com OmniVoice em blocos sequenciais com normalização e limpeza contínua de VRAM."""
    final_tensor = None
    for event_type, msg, tensor in generate_omnivoice_chunked_stream(model, text, ref_audio, ref_text, num_steps, speed):
        if event_type == "complete":
            final_tensor = tensor
    return final_tensor if final_tensor is not None else torch.zeros((1, 24000))

QWEN_LANGUAGES = {
    'pt-br': 'Portuguese',
    'pt': 'Portuguese',
    'en': 'English',
    'es': 'Spanish',
    'fr': 'French',
    'de': 'German',
    'zh-cn': 'Chinese',
    'zh': 'Chinese',
    'it': 'Italian',
    'ja': 'Japanese',
    'ru': 'Russian',
}

def generate_qwen3_tts_chunked_stream(
    model,
    text,
    ref_audio,
    ref_text,
    lang_code='en',
    speed=1.0
):
    """
    Gerador streaming para síntese com Qwen3-TTS 1.7B Base.
    Garante Neutral American Accent e Standard Pronunciation para áudios em inglês
    desacoplando o sotaque estrangeiro do áudio de referência (x_vector_only_mode=True)
    e injetando diretivas de instrução acústica.
    """
    target_clean = lang_code.lower().replace('_', '-')
    qwen_lang = QWEN_LANGUAGES.get(target_clean, 'English')

    is_english = (qwen_lang == 'English')
    use_xvec_only = True if is_english else (False if (ref_text and len(ref_text.strip()) > 0) else True)

    # Injeção de instrução acústica para Neutral American Accent
    instruct_ids = None
    if is_english:
        try:
            if hasattr(model, '_build_instruct_text') and hasattr(model, '_tokenize_texts'):
                ins_text = model._build_instruct_text("Neutral American Accent, Standard Pronunciation")
                instruct_ids = model._tokenize_texts([ins_text])
        except Exception as ins_err:
            print(f"Aviso ao compilar diretiva de sotaque americano: {ins_err}")
            instruct_ids = None

    voice_prompt = None
    try:
        if hasattr(model, 'create_voice_clone_prompt'):
            voice_prompt = model.create_voice_clone_prompt(
                ref_audio=ref_audio,
                ref_text=None if use_xvec_only else (ref_text if ref_text else None),
                x_vector_only_mode=use_xvec_only
            )
    except Exception as prompt_err:
        print(f'Aviso no prompt acústico do Qwen3: {prompt_err}. Utilizando parâmetros diretos.')
        voice_prompt = None

    chunks = split_text_into_chunks(text, max_chunk_chars=260)
    tensors = []
    silence = torch.zeros((1, int(24000 * 0.08)))

    header_msg = f"  ⚡ [Qwen3-TTS 1.7B | {qwen_lang}] Texto dividido em {len(chunks)} blocos para síntese:"
    print(f"\n{header_msg}")
    yield "header", header_msg, None

    for idx, c in enumerate(chunks):
        if is_stop_requested():
            print("\n⏹️ Interrupção solicitada pelo usuário!")
            raise InterruptedError("Operação cancelada pelo usuário.")

        preview = (c[:40] + '...') if len(c) > 40 else c
        start_line = f"    ▶️ Bloco [{idx+1}/{len(chunks)}] ({len(c)} chars): \"{preview}\" "
        print(start_line, end="", flush=True)
        yield "chunk_start", start_line, None

        t0 = time.time()
        gen_kwargs = {
            'text': c,
            'language': qwen_lang
        }
        if instruct_ids is not None:
            gen_kwargs['instruct_ids'] = instruct_ids

        if voice_prompt is not None:
            gen_kwargs['voice_clone_prompt'] = voice_prompt
        else:
            gen_kwargs['ref_audio'] = ref_audio
            gen_kwargs['x_vector_only_mode'] = use_xvec_only
            if ref_text and not use_xvec_only:
                gen_kwargs['ref_text'] = ref_text

        out = model.generate_voice_clone(**gen_kwargs)
        wavs, sr = out if isinstance(out, tuple) else (out, 24000)

        raw = wavs[0] if isinstance(wavs, (list, tuple)) else wavs
        if isinstance(raw, torch.Tensor):
            t = raw.detach().cpu().float()
        else:
            t = torch.from_numpy(np.array(raw, dtype=np.float32))

        if t.dim() == 1:
            t = t.unsqueeze(0)

        if sr != 24000:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=24000)
            t = resampler(t)

        t = normalize_audio_tensor(t)
        tensors.append(t)
        tensors.append(silence)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        elapsed = time.time() - t0
        done_line = f"{start_line}✅ ({elapsed:.1f}s)"
        print(f"✅ ({elapsed:.1f}s)")
        yield "chunk_done", done_line, t

    if tensors:
        tensors.pop()
        combined = torch.cat(tensors, dim=-1)
        yield "complete", None, normalize_audio_tensor(combined)
    else:
        yield "complete", None, torch.zeros((1, 24000))

def generate_qwen3_tts_chunked(
    model,
    text,
    ref_audio,
    ref_text,
    lang_code='en',
    speed=1.0,
    progress_callback=None
):
    """Sintetiza fala com o Qwen3-TTS 1.7B Base de forma síncrona."""
    final_tensor = None
    for event_type, msg, tensor in generate_qwen3_tts_chunked_stream(
        model=model,
        text=text,
        ref_audio=ref_audio,
        ref_text=ref_text,
        lang_code=lang_code,
        speed=speed
    ):
        if event_type == "complete":
            final_tensor = tensor
    return final_tensor if final_tensor is not None else torch.zeros((1, 24000))

def synthesize_speech_sequential(
    model_choice,
    text,
    ref_audio,
    ref_text,
    lang_code='en',
    num_steps=32,
    speed=1.0,
    progress_callback=None
):
    """
    Sintetiza um idioma por vez utilizando exclusivamente o modelo selecionado na GPU.
    Garante limpeza de VRAM imediata após a geração do áudio.
    """
    engine_name, model = load_tts_model(model_choice)

    try:
        if engine_name == 'Qwen3-TTS':
            audio_tensor = generate_qwen3_tts_chunked(
                model=model,
                text=text,
                ref_audio=ref_audio,
                ref_text=ref_text,
                lang_code=lang_code,
                speed=speed,
                progress_callback=progress_callback
            )
        else:
            audio_tensor = generate_omnivoice_chunked(
                model=model,
                text=text,
                ref_audio=ref_audio,
                ref_text=ref_text,
                num_steps=num_steps,
                speed=speed,
                progress_callback=progress_callback
            )
        return audio_tensor
    finally:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

def process_srt_cloning(
    srt_file_path,
    ref_audio,
    model_engine,
    lang_code='pt-BR',
    num_steps=32,
    speed=1.0,
    progress_callback=None
):
    """
    Sintetiza fala clonada seguindo exatamente as marcações temporais de uma legenda .SRT.
    Insere silêncio correspondente às pausas entre as legendas e ajusta o tempo de cada fala.
    """
    entries = parse_srt(srt_file_path)
    if not entries:
        raise ValueError("O arquivo SRT está vazio ou possui formatação incompatível.")

    total_subs = len(entries)
    print(f"\n📄 [SRT Sync] {total_subs} legendas detectadas. Iniciando síntese alinhada...")

    ref_wav = tempfile.NamedTemporaryFile(suffix='_srt_ref.wav', delete=False).name
    extract_audio_to_wav(ref_audio, ref_wav)
    ref_asr = whisper_model.transcribe(ref_wav, task='transcribe')
    ref_text = ref_asr.get('text', '').strip()

    timeline_parts = []
    current_time = 0.0

    for idx, item in enumerate(entries):
        if is_stop_requested():
            print("\n⏹️ Interrupção solicitada pelo usuário!")
            raise InterruptedError("Operação cancelada pelo usuário.")

        if progress_callback:
            progress_callback(idx, total_subs, item['text'], status="running", elapsed=None)

        start_time = item['start']
        end_time = item['end']
        target_dur = max(0.4, end_time - start_time)

        # Insere silêncio exato se houver intervalo antes da próxima legenda
        if start_time > current_time + 0.03:
            gap = start_time - current_time
            silence_samples = int(24000 * gap)
            timeline_parts.append(torch.zeros((1, silence_samples)))
            current_time = start_time

        t0 = time.time()
        # Síntese do bloco de texto da legenda
        sub_audio = synthesize_speech_sequential(
            model_choice=model_engine,
            text=item['text'],
            ref_audio=ref_wav,
            ref_text=ref_text,
            lang_code=lang_code,
            num_steps=int(num_steps),
            speed=float(speed)
        )

        temp_sub_wav = tempfile.NamedTemporaryFile(suffix='_sub_raw.wav', delete=False).name
        torchaudio.save(temp_sub_wav, sub_audio.cpu(), 24000)

        temp_sub_synced = tempfile.NamedTemporaryFile(suffix='_sub_synced.wav', delete=False).name
        time_sync_audio(temp_sub_wav, target_dur, temp_sub_synced, max_stretch=0.20)

        t_synced, _ = torchaudio.load(temp_sub_synced)
        t_synced = normalize_audio_tensor(t_synced, target_peak=0.95)
        timeline_parts.append(t_synced)
        current_time = end_time

        elapsed = time.time() - t0
        if progress_callback:
            progress_callback(idx, total_subs, item['text'], status="done", elapsed=elapsed)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        for p in (temp_sub_wav, temp_sub_synced):
            if os.path.exists(p):
                os.remove(p)

    if os.path.exists(ref_wav):
        os.remove(ref_wav)

    if not timeline_parts:
        return torch.zeros((1, 24000))

    final_tensor = torch.cat(timeline_parts, dim=-1)
    return normalize_audio_tensor(final_tensor)

print('✅ Motor de processamento, síntese sequencial streaming, SRT sync, STOP handler e limpeza de VRAM carregados com sucesso!')


In [ ]:
# @title Passo 4: Iniciar a Interface de Dublagem & Clonagem de Áudio (Gradio)
# @markdown Clique no botão 'Play' e acesse o link público 'Running on public URL: https://...gradio.live'

import os
import re
import shutil
import tempfile
import gc
import time
import glob
import threading
import json
import torch
import torchaudio
import gradio as gr

OUTPUTS_DIR = os.path.abspath("./outputs")
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# ==============================================================================
# 🛡️ SENTINELA ANTI-TIMEOUT & SESSÃO PERSISTENTE (GOOGLE COLAB)
# ==============================================================================
try:
    from IPython.display import display, Javascript
    colab_keepalive_script = """
    (function() {
        if (window.__matraca_colab_keepalive_active) return;
        window.__matraca_colab_keepalive_active = true;
        console.log("🛡️ [Matraca KeepAlive] Ativando sentinela anti-timeout no Google Colab...");
        function pingColab() {
            try {
                const btnShadow = document.querySelector("colab-connect-button")?.shadowRoot?.getElementById("connect");
                if (btnShadow) {
                    btnShadow.click();
                    console.log("[Matraca KeepAlive] Ping Colab (shadowRoot) enviado: " + new Date().toLocaleTimeString());
                }
                const btnToolbar = document.querySelector("#connect") || document.querySelector("colab-toolbar-button#connect");
                if (btnToolbar) {
                    btnToolbar.click();
                    console.log("[Matraca KeepAlive] Ping Colab (toolbar) enviado: " + new Date().toLocaleTimeString());
                }
            } catch (e) {}
        }
        setInterval(pingColab, 60000);
        console.log("✅ [Matraca KeepAlive] Sentinela Colab operando (intervalo: 60s).");
    })();
    """
    display(Javascript(colab_keepalive_script))
    print("🛡️ Sentinela Anti-Timeout ativado com sucesso para o Google Colab (ping a cada 60s)!")
except Exception as e:
    pass

def get_existing_outputs():
    """Retorna a lista de todos os arquivos WAV já salvos na pasta outputs."""
    return sorted(glob.glob(os.path.join(OUTPUTS_DIR, "*.wav")), key=os.path.getmtime, reverse=True)

def create_zip_outputs():
    """Gera um arquivo ZIP com todos os áudios presentes na pasta outputs."""
    zip_path = os.path.abspath("./audios_dublados.zip")
    if os.path.exists(zip_path):
        try:
            os.remove(zip_path)
        except Exception:
            pass
    wav_files = glob.glob(os.path.join(OUTPUTS_DIR, "*.wav"))
    if not wav_files:
        return None
    shutil.make_archive(os.path.splitext(zip_path)[0], 'zip', OUTPUTS_DIR)
    return zip_path if os.path.exists(zip_path) else None

# Opções de idiomas para o áudio original com detecção robusta
ORIG_LANG_CHOICES = {
    '🇧🇷 Português (pt-BR)': 'pt',
    '🌐 Detectar Automaticamente': 'auto',
    '🇺🇸 Inglês (English)': 'en',
    '🇪🇸 Espanhol (Español)': 'es',
    '🇫🇷 Francês (Français)': 'fr',
    '🇩🇪 Alemão (Deutsch)': 'de',
    '🇨🇳 Chinês Simplificado (中文)': 'zh',
    '🇮🇹 Italiano (Italiano)': 'it',
    '🇯🇵 Japonês (日本語)': 'ja',
    '🇷🇺 Russo (Русский)': 'ru'
}

# Todos os idiomas da interface
ALL_LANGUAGES = {
    '🇧🇷 Português Brasileiro (pt-BR)': 'pt-BR',
    '🇺🇸 Inglês (English)': 'en',
    '🇪🇸 Espanhol (Español)': 'es',
    '🇫🇷 Francês (Français)': 'fr',
    '🇩🇪 Alemão (Deutsch)': 'de',
    '🇨🇳 Chinês Simplificado (中文)': 'zh-CN',
    '🇮🇹 Italiano (Italiano)': 'it',
    '🇯🇵 Japonês (日本語)': 'ja',
    '🇷🇺 Russo (Русский)': 'ru',
    '🇸🇦 Árabe (العربية)': 'ar'
}

# Qwen3-TTS suporta 9 idiomas (Árabe não é suportado pelo Qwen3-TTS)
QWEN_SUPPORTED_LABELS = [
    '🇧🇷 Português Brasileiro (pt-BR)',
    '🇺🇸 Inglês (English)',
    '🇪🇸 Espanhol (Español)',
    '🇫🇷 Francês (Français)',
    '🇩🇪 Alemão (Deutsch)',
    '🇨🇳 Chinês Simplificado (中文)',
    '🇮🇹 Italiano (Italiano)',
    '🇯🇵 Japonês (日本語)',
    '🇷🇺 Russo (Русский)'
]

OMNIVOICE_SUPPORTED_LABELS = list(ALL_LANGUAGES.keys())

def resolve_lang_code(lang_label):
    if lang_label in ALL_LANGUAGES:
        return ALL_LANGUAGES[lang_label]
    l_lower = lang_label.lower()
    if 'espanh' in l_lower or 'es' in l_lower:
        return 'es'
    if 'ingl' in l_lower or 'en' in l_lower:
        return 'en'
    if 'portug' in l_lower or 'pt' in l_lower:
        return 'pt-BR'
    if 'franc' in l_lower or 'fr' in l_lower:
        return 'fr'
    if 'alem' in l_lower or 'de' in l_lower:
        return 'de'
    if 'chin' in l_lower or 'zh' in l_lower:
        return 'zh-CN'
    if 'arab' in l_lower or 'ar' in l_lower:
        return 'ar'
    if 'ital' in l_lower or 'it' in l_lower:
        return 'it'
    if 'japon' in l_lower or 'ja' in l_lower:
        return 'ja'
    if 'russ' in l_lower or 'ru' in l_lower:
        return 'ru'
    return 'en'

def on_model_selection_changed(model_choice, current_selected):
    """
    Atualiza dinamicamente as caixas de seleção de idiomas:
    - Se Qwen3-TTS for selecionado: Árabe é removido das escolhas e desmarcado.
    - Se OmniVoice for selecionado: Todos os 10 idiomas (incluindo Árabe) são liberados.
    """
    is_qwen = 'Qwen' in str(model_choice)
    choices = QWEN_SUPPORTED_LABELS if is_qwen else OMNIVOICE_SUPPORTED_LABELS
    valid_selection = [lang for lang in (current_selected or []) if lang in choices]
    if not valid_selection:
        valid_selection = ['🇺🇸 Inglês (English)', '🇪🇸 Espanhol (Español)']
    return gr.update(choices=choices, value=valid_selection)

# ==============================================================================
# 🚀 JOBMANAGER: GERENCIADOR DE PROCESSAMENTO EM SEGUNDO PLANO (RESISTENTE A F5)
# ==============================================================================
class JobManager:
    """
    Gerenciador global de execução em segundo plano para o Matraca Studio.
    
    Principais garantias:
    - Execução desacoplada em threads em segundo plano (imune a F5 ou desconexão do navegador).
    - Persistência contínua de status e logs em ./outputs/job_state.json e ./outputs/latest_job.log.
    - Reconexão transparente: qualquer cliente que der refresh na página recupera estado e logs.
    - Cancelamento seguro e imediato com liberação de cache de GPU VRAM.
    """
    _instance = None
    _lock = threading.Lock()

    def __new__(cls, *args, **kwargs):
        if not cls._instance:
            with cls._lock:
                if not cls._instance:
                    cls._instance = super(JobManager, cls).__new__(cls)
                    cls._instance._init_manager()
        return cls._instance

    def _init_manager(self):
        self.lock = threading.RLock()
        self.active_thread = None
        self.state_file = os.path.join(OUTPUTS_DIR, "job_state.json")
        self.log_file = os.path.join(OUTPUTS_DIR, "latest_job.log")
        self.reset_state(persist=False)
        self._load_persisted_state()

    def reset_state(self, persist=True):
        with self.lock:
            self.job_id = None
            self.job_type = None  # 'dubbing' | 'free_cloning' | None
            self.status = 'idle'  # 'idle' | 'running' | 'completed' | 'error' | 'cancelled'
            self.stage = 'Pronto para processar'
            self.start_time = None
            self.end_time = None
            self.logs = []
            self.preview_audio = None
            self.output_files = []
            self.translations_summary = ''
            self.status_report = ''
            self.error_message = ''
            self.workflow_step = 0
            if persist:
                self._persist_state()

    def _persist_state(self):
        try:
            with self.lock:
                elapsed = round(time.time() - self.start_time, 1) if self.start_time else 0
                data = {
                    "job_id": self.job_id,
                    "job_type": self.job_type,
                    "status": self.status,
                    "stage": self.stage,
                    "start_time": self.start_time,
                    "end_time": self.end_time,
                    "elapsed": elapsed,
                    "preview_audio": self.preview_audio,
                    "output_files": self.output_files,
                    "translations_summary": self.translations_summary,
                    "status_report": self.status_report,
                    "error_message": self.error_message,
                    "workflow_step": self.workflow_step,
                    "log_count": len(self.logs),
                    "last_updated": time.time()
                }
                with open(self.state_file, "w", encoding="utf-8") as f:
                    json.dump(data, f, ensure_ascii=False, indent=2)
        except Exception as e:
            print(f"[JobManager] Erro ao salvar estado: {e}")

    def _load_persisted_state(self):
        try:
            if os.path.exists(self.state_file):
                with open(self.state_file, "r", encoding="utf-8") as f:
                    data = json.load(f)
                with self.lock:
                    self.job_id = data.get("job_id")
                    self.job_type = data.get("job_type")
                    persisted_status = data.get("status", "idle")
                    if persisted_status == "running":
                        self.status = "error"
                        self.stage = "Processo reiniciado antes da conclusão"
                    else:
                        self.status = persisted_status
                        self.stage = data.get("stage", "Pronto para processar")
                    self.start_time = data.get("start_time")
                    self.end_time = data.get("end_time")
                    self.preview_audio = data.get("preview_audio")
                    self.output_files = data.get("output_files", [])
                    self.translations_summary = data.get("translations_summary", "")
                    self.status_report = data.get("status_report", "")
                    self.error_message = data.get("error_message", "")
                    self.workflow_step = data.get("workflow_step", 0)
                if os.path.exists(self.log_file):
                    with open(self.log_file, "r", encoding="utf-8") as lf:
                        self.logs = lf.read().splitlines()
        except Exception as e:
            print(f"[JobManager] Erro ao ler estado persistido: {e}")

    def add_log(self, msg):
        if not msg:
            return
        lines = msg.splitlines() if isinstance(msg, str) else [str(msg)]
        with self.lock:
            for l in lines:
                self.logs.append(l)
            try:
                with open(self.log_file, "a", encoding="utf-8") as f:
                    for l in lines:
                        f.write(l + "\n")
            except Exception:
                pass
        for l in lines:
            print(f"[MATRACA] {l}")

    def update_stage(self, stage=None, workflow_step=None, preview=None, files=None, translations=None, report=None):
        with self.lock:
            if stage is not None:
                self.stage = stage
            if workflow_step is not None:
                self.workflow_step = workflow_step
            if preview is not None:
                self.preview_audio = preview
            if files is not None:
                self.output_files = files
            if translations is not None:
                self.translations_summary = translations
            if report is not None:
                self.status_report = report
            self._persist_state()

    def is_running(self):
        with self.lock:
            return self.status == 'running' and self.active_thread is not None and self.active_thread.is_alive()

    def start_job(self, job_type, worker_fn, *args, **kwargs):
        with self.lock:
            if self.is_running():
                return False, f"Já existe uma tarefa em execução ({self.job_id}). Aguarde ou clique em Cancelar."

            self.reset_state(persist=False)
            set_stop_requested(False)
            self.job_id = f"job_{job_type}_{int(time.time())}"
            self.job_type = job_type
            self.status = 'running'
            self.stage = 'Iniciando processamento em segundo plano...'
            self.start_time = time.time()
            self.workflow_step = 4

            try:
                with open(self.log_file, "w", encoding="utf-8") as f:
                    f.write(f"=== {self.job_id} [{job_type.upper()}] INICIADO EM {time.ctime()} ===\n")
            except Exception:
                pass

            self._persist_state()

            self.active_thread = threading.Thread(
                target=self._worker_wrapper,
                args=(worker_fn, args, kwargs),
                daemon=True,
                name=f"Worker_{self.job_id}"
            )
            self.active_thread.start()
            return True, self.job_id

    def _worker_wrapper(self, worker_fn, args, kwargs):
        try:
            worker_fn(self, *args, **kwargs)
            with self.lock:
                if self.status == 'running':
                    self.status = 'completed'
                    self.stage = 'Processamento concluído com sucesso!'
                    self.end_time = time.time()
                    self.workflow_step = 5
                    self.add_log("\n🎉 Processamento finalizado com êxito!")
                    self._persist_state()
        except InterruptedError:
            with self.lock:
                self.status = 'cancelled'
                self.stage = 'Operação cancelada pelo usuário.'
                self.end_time = time.time()
                self.add_log("\n⏹️ Operação cancelada pelo usuário! Cache de VRAM liberado.")
                self._persist_state()
        except Exception as e:
            with self.lock:
                self.status = 'error'
                self.stage = f"Falha: {str(e)}"
                self.error_message = str(e)
                self.end_time = time.time()
                self.add_log(f"\n❌ Erro durante processamento: {str(e)}")
                self._persist_state()
        finally:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
            gc.collect()

    def request_cancel(self):
        with self.lock:
            set_stop_requested(True)
            if self.is_running():
                self.stage = 'Interrupção solicitada pelo usuário... Encerrando com segurança...'
                self.add_log("⚠️ Solicitação de interrupção recebida. Encerrando bloco atual e liberando GPU...")
                self._persist_state()
                return "⏹️ Interrupção solicitada! Encerrando processo e liberando cache da GPU..."
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
            gc.collect()
            return "Nenhuma tarefa em execução para cancelar."

    def get_state(self):
        with self.lock:
            elapsed = round(time.time() - self.start_time, 1) if self.start_time else 0
            return {
                "job_id": self.job_id,
                "job_type": self.job_type,
                "status": self.status,
                "stage": self.stage,
                "elapsed": elapsed,
                "logs_text": "\n".join(self.logs),
                "preview_audio": self.preview_audio,
                "output_files": list(self.output_files),
                "translations_summary": self.translations_summary,
                "status_report": self.status_report,
                "error_message": self.error_message,
                "workflow_step": self.workflow_step,
                "is_running": self.is_running()
            }

job_manager = JobManager()

def render_job_monitor_banner(state=None):
    if state is None:
        state = job_manager.get_state()
    status = state.get("status", "idle")
    stage = state.get("stage", "Pronto para processar")
    elapsed = state.get("elapsed", 0)
    job_id = state.get("job_id", "-")

    if status == 'running':
        return f"""<div class='matraca-monitor-card running'>
            <div style='display:flex;align-items:center;justify-content:space-between;flex-wrap:wrap;gap:8px;'>
                <span class='monitor-status-badge running'>⚡ Em Execução em Segundo Plano</span>
                <span class='monitor-note'>⏱️ Decorrido: <strong>{elapsed:.1f}s</strong> | ID: <code>{job_id}</code></span>
            </div>
            <div class='monitor-info'><strong>Etapa Atual:</strong> {stage}</div>
            <div class='monitor-note'>🛡️ <em>Execução desacoplada: recarregar a página (F5) ou oscilações de rede não interrompem esta tarefa.</em></div>
        </div>"""
    elif status == 'completed':
        return f"""<div class='matraca-monitor-card completed'>
            <div style='display:flex;align-items:center;justify-content:space-between;flex-wrap:wrap;gap:8px;'>
                <span class='monitor-status-badge completed'>✅ Último Job Concluído com Sucesso</span>
                <span class='monitor-note'>Duração: <strong>{elapsed:.1f}s</strong> | ID: <code>{job_id}</code></span>
            </div>
            <div class='monitor-info'>{stage}</div>
        </div>"""
    elif status == 'cancelled':
        return f"""<div class='matraca-monitor-card cancelled'>
            <div style='display:flex;align-items:center;justify-content:space-between;flex-wrap:wrap;gap:8px;'>
                <span class='monitor-status-badge cancelled'>⏹️ Job Cancelado pelo Usuário</span>
                <span class='monitor-note'>ID: <code>{job_id}</code></span>
            </div>
            <div class='monitor-info'>{stage} (Cache de GPU/VRAM liberado com sucesso)</div>
        </div>"""
    elif status == 'error':
        return f"""<div class='matraca-monitor-card cancelled'>
            <div style='display:flex;align-items:center;justify-content:space-between;flex-wrap:wrap;gap:8px;'>
                <span class='monitor-status-badge cancelled'>❌ Falha na Execução</span>
                <span class='monitor-note'>ID: <code>{job_id}</code></span>
            </div>
            <div class='monitor-info'><strong>Detalhes:</strong> {state.get('error_message', stage)}</div>
        </div>"""
    else:
        return """<div class='matraca-monitor-card idle'>
            <div style='display:flex;align-items:center;justify-content:space-between;flex-wrap:wrap;gap:8px;'>
                <span class='monitor-status-badge idle'>🟢 Servidor Pronto / Ocioso</span>
                <span class='monitor-note'>🛡️ Sentinela Anti-Timeout Ativo</span>
            </div>
            <div class='monitor-info'>Envie seu áudio ou texto para iniciar a dublagem ou clonagem vocal.</div>
        </div>"""

def stop_process():
    """Aciona a interrupção global e limpa imediatamente o cache de VRAM e memória."""
    return job_manager.request_cancel()

def transcribe_only(audio_file, orig_lang_label):
    """Transcreve o áudio original com Whisper usando o idioma selecionado explicitamente."""
    if not audio_file:
        return '', '⚠️ Por favor, envie ou grave um arquivo de áudio antes de transcrever.'
    try:
        orig_dur = get_audio_duration(audio_file)
        temp_wav = tempfile.NamedTemporaryFile(suffix='_transcribe.wav', delete=False).name
        extract_audio_to_wav(audio_file, temp_wav)

        lang_code = ORIG_LANG_CHOICES.get(orig_lang_label, 'pt')
        lang_param = None if lang_code == 'auto' else lang_code

        # Força task='transcribe' para impedir tradução forçada para o inglês
        asr_res = whisper_model.transcribe(temp_wav, language=lang_param, task='transcribe')
        text = asr_res.get('text', '').strip()
        lang = asr_res.get('language', lang_code if lang_code != 'auto' else 'desconhecido')
        segments = asr_res.get('segments', [])

        speech_start = float(segments[0]['start']) if segments else 0.0

        if os.path.exists(temp_wav):
            os.remove(temp_wav)

        intro_text = f" | ⏱️ **Intro detectada:** `{speech_start:.2f}s`" if speech_start >= 0.25 else ""
        status_msg = f"""✅ **Transcrição concluída com sucesso!**
- ⏱️ **Duração do Áudio:** `{orig_dur:.2f}s` | 🌐 **Idioma utilizado:** `{lang.upper()}`{intro_text}
> 💡 *Você pode editar o texto abaixo, escolher os idiomas e clicar em **'▶ Dublar e sincronizar'**.*"""
        return text, status_msg
    except Exception as e:
        return '', f'❌ **Erro ao transcrever áudio:** `{str(e)}`'

# ==============================================================================
# 🧵 WORKER DE DUBLAGEM (EXECUTADO EM SEGUNDO PLANO)
# ==============================================================================
def run_dubbing_worker(
    jm,
    audio_file,
    orig_lang_label,
    edited_transcription,
    filtered_langs,
    custom_ref_audio,
    sync_duration_opt,
    num_steps,
    user_speed,
    model_engine
):
    total_langs = len(filtered_langs)
    is_qwen = 'Qwen' in str(model_engine)
    engine_label = 'Qwen3-TTS 1.7B' if is_qwen else 'OmniVoice'

    header_start = f"🚀 Iniciando dublagem sequencial com [{engine_label}] para {total_langs} idioma(s): {filtered_langs}"
    jm.add_log(header_start)
    jm.update_stage("Analisando áudio e preparando linha do tempo...", workflow_step=4)

    temp_full_wav = None
    ref_slice_wav = None
    try:
        orig_duration = get_audio_duration(audio_file)
        temp_full_wav = tempfile.NamedTemporaryFile(suffix='_full.wav', delete=False).name
        extract_audio_to_wav(audio_file, temp_full_wav)

        lang_code_input = ORIG_LANG_CHOICES.get(orig_lang_label, 'pt')
        lang_param = None if lang_code_input == 'auto' else lang_code_input

        asr_result = whisper_model.transcribe(temp_full_wav, language=lang_param, task='transcribe')
        whisper_text = asr_result.get('text', '').strip()
        segments = asr_result.get('segments', [])
        source_lang_code = asr_result.get('language', lang_code_input) or lang_code_input

        original_text = (edited_transcription or '').strip()
        if not original_text:
            original_text = whisper_text

        if not original_text:
            raise ValueError("Não foi possível reconhecer fala audível no arquivo enviado.")

        speech_start = float(segments[0]['start']) if segments else 0.0
        speech_end = float(segments[-1]['end']) if segments else orig_duration

        ref_slice_wav = tempfile.NamedTemporaryFile(suffix='_ref_slice.wav', delete=False).name
        if custom_ref_audio and os.path.exists(custom_ref_audio):
            jm.add_log("🎙️ Utilizando áudio de referência limpo enviado pelo usuário!")
            extract_audio_to_wav(custom_ref_audio, ref_slice_wav)
            ref_asr = whisper_model.transcribe(ref_slice_wav, language=lang_param, task='transcribe')
            ref_slice_text = ref_asr.get('text', '').strip()
        else:
            jm.add_log("🔍 Selecionando trecho vocal mais limpo da mídia original...")
            _, _, ref_slice_text = select_best_voice_slice(
                segments=segments,
                full_wav=temp_full_wav,
                orig_duration=orig_duration,
                output_slice_wav=ref_slice_wav,
                whisper_model=whisper_model,
                orig_lang=lang_param
            )

        orig_base_name = os.path.splitext(os.path.basename(audio_file))[0]
        clean_base_name = re.sub(r'[^a-zA-Z0-9_\-]', '_', orig_base_name)

        generated_files = []
        translations_summary = []
        results_info = []
        first_preview_audio = None

        # Loop sequencial por idioma
        for idx, lang_label in enumerate(filtered_langs):
            if is_stop_requested():
                raise InterruptedError()

            lang_code = resolve_lang_code(lang_label)
            clean_code = lang_code.replace('-', '_')

            # ETAPA 1: Tradução
            trans_start_msg = f"\n════════════════════════════════════════════════════════\n🌐 [{idx+1}/{total_langs}] Idioma: {lang_label}\n⏳ Traduzindo e validando texto antes da síntese vocal..."
            jm.add_log(trans_start_msg)
            jm.update_stage(f"Traduzindo para {lang_label} ({idx+1}/{total_langs})...")

            translated_text = translate_text_robust(
                original_text,
                lang_code,
                source_code=source_lang_code,
                max_chunk=700,
                progress_callback=lambda m: jm.add_log(m)
            )
            translations_summary.append(f'### {lang_label}\n{translated_text}\n')

            trans_ok_msg = f"✅ Tradução validada com sucesso ({len(translated_text)} caracteres)!\n🎙️ Preparando modelo [{engine_label}] para síntese vocal..."
            jm.add_log(trans_ok_msg)
            jm.update_stage(f"Síntese vocal de {lang_label} ({idx+1}/{total_langs})...", translations='\n'.join(translations_summary))

            # ETAPA 2: Síntese Vocal
            engine_name, model = load_tts_model(model_engine)

            stream_gen = (
                generate_qwen3_tts_chunked_stream(
                    model=model,
                    text=translated_text,
                    ref_audio=ref_slice_wav,
                    ref_text=ref_slice_text,
                    lang_code=lang_code,
                    speed=float(user_speed)
                )
                if engine_name == 'Qwen3-TTS'
                else generate_omnivoice_chunked_stream(
                    model=model,
                    text=translated_text,
                    ref_audio=ref_slice_wav,
                    ref_text=ref_slice_text,
                    num_steps=int(num_steps),
                    speed=float(user_speed)
                )
            )

            audio_tensor = None
            for event_type, msg, t_data in stream_gen:
                if is_stop_requested():
                    raise InterruptedError()

                if event_type == "header":
                    jm.add_log("\n" + msg)
                    jm.update_stage(f"Dublando para {lang_label} ({idx+1}/{total_langs})...")
                elif event_type == "chunk_done":
                    jm.add_log(msg)
                elif event_type == "complete":
                    audio_tensor = t_data

            if audio_tensor is None or is_stop_requested():
                raise InterruptedError()

            temp_synth_wav = tempfile.NamedTemporaryFile(suffix='_synth.wav', delete=False).name
            torchaudio.save(temp_synth_wav, audio_tensor.cpu(), 24000)

            final_audio_path = os.path.join(OUTPUTS_DIR, f'{clean_base_name}_{clean_code}_dublado.wav')
            speed_factor, final_duration = assemble_timeline_audio(
                original_audio_path=temp_full_wav,
                synth_speech_wav_path=temp_synth_wav,
                speech_start=speech_start,
                speech_end=speech_end,
                total_duration=orig_duration,
                output_final_wav=final_audio_path,
                sync_duration=sync_duration_opt
            )
            generated_files.append(final_audio_path)
            if first_preview_audio is None:
                first_preview_audio = final_audio_path

            results_info.append({
                'label': lang_label,
                'code': lang_code,
                'final_dur': final_duration,
                'speed_factor': speed_factor,
                'audio_file': os.path.basename(final_audio_path)
            })

            if os.path.exists(temp_synth_wav):
                os.remove(temp_synth_wav)

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

            jm.update_stage(
                stage=f"Sincronizado {lang_label} ({idx+1}/{total_langs})",
                preview=first_preview_audio,
                files=list(generated_files),
                translations='\n'.join(translations_summary)
            )

        # Monta relatório final
        rows_md = []
        for r in results_info:
            tempo_col = f"{r['speed_factor']:.2f}x" if sync_duration_opt else 'Original'
            rows_md.append(f"| **{r['label']}** | `{r['audio_file']}` | `{tempo_col}` | `{r['final_dur']:.2f}s` |")
        table_body = '\n'.join(rows_md)

        status_report = f"""### 🎉 Dublagem de Áudio Concluída com Sucesso!
**Motor de IA Utilizado:** `{engine_label}`

| Idioma | Áudio WAV Sincronizado | Ajuste de Tempo | Duração Final |
| :--- | :--- | :---: | :---: |
{table_body}

> ℹ️ *Acesse os arquivos gerados abaixo para reprodução direta ou download individual imediato.*"""

        jm.update_stage(
            stage="Processamento concluído com sucesso!",
            workflow_step=5,
            preview=first_preview_audio,
            files=generated_files,
            translations='\n'.join(translations_summary),
            report=status_report
        )

    finally:
        for p in (temp_full_wav, ref_slice_wav):
            if p and os.path.exists(p):
                try:
                    os.remove(p)
                except Exception:
                    pass

def process_dubbing(
    audio_file,
    orig_lang_label,
    edited_transcription,
    target_lang_labels,
    custom_ref_audio,
    sync_duration_opt,
    num_steps,
    user_speed,
    model_engine
):
    """
    Inicia a dublagem no JobManager em segundo plano e realiza streaming dos resultados.
    Se a conexão cair ou a página for atualizada (F5), o trabalho continua sem interrupções.
    """
    if not audio_file:
        yield [], '❌ **Erro:** Por favor, envie ou grave um arquivo de áudio (.wav, .mp3, .m4a, etc.).', '❌ Erro: Nenhum arquivo de áudio enviado.', '', render_job_monitor_banner()
        return

    if not target_lang_labels or len(target_lang_labels) == 0:
        yield [], '❌ **Erro:** Por favor, marque pelo menos um idioma de destino nas caixas de seleção.', '❌ Erro: Nenhum idioma de destino selecionado.', '', render_job_monitor_banner()
        return

    is_qwen = 'Qwen' in str(model_engine)
    filtered_langs = []
    for l in target_lang_labels:
        code = resolve_lang_code(l)
        if is_qwen and code == 'ar':
            print("⚠️ Árabe desconsiderado: Qwen3-TTS não possui suporte nativo ao idioma Árabe.")
            continue
        filtered_langs.append(l)

    if not filtered_langs:
        yield [], '❌ **Erro:** O modelo Qwen3-TTS não suporta o idioma Árabe. Selecione o OmniVoice para dublar em Árabe.', '❌ Erro: Idioma Árabe não suportado pelo Qwen3-TTS.', '', render_job_monitor_banner()
        return

    if not job_manager.is_running():
        ok, job_id = job_manager.start_job(
            'dubbing',
            run_dubbing_worker,
            audio_file=audio_file,
            orig_lang_label=orig_lang_label,
            edited_transcription=edited_transcription,
            filtered_langs=filtered_langs,
            custom_ref_audio=custom_ref_audio,
            sync_duration_opt=sync_duration_opt,
            num_steps=num_steps,
            user_speed=user_speed,
            model_engine=model_engine
        )
        if not ok:
            st = job_manager.get_state()
            yield st['output_files'], f"⚠️ {job_id}", st['logs_text'], st['translations_summary'], render_job_monitor_banner(st)
            return

    # Loop de streaming desacoplado
    while job_manager.is_running():
        st = job_manager.get_state()
        status_display = f"### 🟡 Dublagem em Execução em Segundo Plano\n- **ID:** `{st['job_id']}` | ⏱️ **Decorrido:** `{st['elapsed']:.1f}s`\n- **Etapa:** {st['stage']}\n\n> 🛡️ *Este processo continua ativo mesmo se você fechar ou recarregar a página (F5).*"
        yield st['output_files'], status_display, st['logs_text'], st['translations_summary'], render_job_monitor_banner(st)
        time.sleep(0.6)

    # Estado final pós-conclusão
    final_st = job_manager.get_state()
    if final_st['status'] == 'completed':
        yield final_st['output_files'], final_st['status_report'], final_st['logs_text'], final_st['translations_summary'], render_job_monitor_banner(final_st)
    elif final_st['status'] == 'cancelled':
        yield final_st['output_files'], "⚠️ **Processo interrompido pelo usuário.**", final_st['logs_text'], final_st['translations_summary'], render_job_monitor_banner(final_st)
    else:
        yield final_st['output_files'], f"❌ **Falha durante processamento:** `{final_st['error_message']}`", final_st['logs_text'], final_st['translations_summary'], render_job_monitor_banner(final_st)

# ==============================================================================
# 🎤 WORKER DE CLONAGEM LIVRE (EXECUTADO EM SEGUNDO PLANO)
# ==============================================================================
def run_free_cloning_worker(
    jm,
    input_mode,
    custom_text,
    srt_file,
    ref_audio,
    lang_label,
    num_steps,
    speed,
    model_engine
):
    lang_code = resolve_lang_code(lang_label)
    engine_label = 'Qwen3-TTS 1.7B' if 'Qwen' in str(model_engine) else 'OmniVoice'

    if input_mode == '📄 Importar Legenda (.SRT)':
        jm.add_log(f"🎬 Iniciando síntese via legenda SRT com [{engine_label}]...")
        jm.update_stage("Processando legenda SRT e sintetizando áudio...")
        audio_tensor = process_srt_cloning(
            srt_file_path=srt_file,
            ref_audio=ref_audio,
            model_engine=model_engine,
            lang_code=lang_code,
            num_steps=int(num_steps),
            speed=float(speed)
        )
        out_name = f"clonagem_srt_{int(time.time())}.wav"
        final_path = os.path.join(OUTPUTS_DIR, out_name)
        torchaudio.save(final_path, audio_tensor.cpu(), 24000)

        dur = get_audio_duration(final_path)
        report = f'✅ Áudio clonado gerado com sincronia temporal exata da legenda SRT ({dur:.2f}s) via [{engine_label}]!'
        jm.update_stage(stage="Concluído!", preview=final_path, files=[final_path], report=report)
    else:
        jm.add_log(f"✍️ Iniciando síntese de texto livre com [{engine_label}]...")
        jm.update_stage("Analisando amostra de voz de referência...")
        ref_wav = tempfile.NamedTemporaryFile(suffix='_free_ref.wav', delete=False).name
        try:
            extract_audio_to_wav(ref_audio, ref_wav)
            ref_asr = whisper_model.transcribe(ref_wav, task='transcribe')
            ref_text = ref_asr.get('text', '').strip()

            jm.update_stage("Sintetizando fala na GPU...")
            audio_tensor = synthesize_speech_sequential(
                model_choice=model_engine,
                text=custom_text.strip(),
                ref_audio=ref_wav,
                ref_text=ref_text,
                lang_code=lang_code,
                num_steps=int(num_steps),
                speed=float(speed)
            )
            out_name = f"clonagem_livre_{int(time.time())}.wav"
            final_path = os.path.join(OUTPUTS_DIR, out_name)
            torchaudio.save(final_path, audio_tensor.cpu(), 24000)

            report = f'✅ Fala sintetizada com sucesso via [{engine_label}]!'
            jm.update_stage(stage="Concluído!", preview=final_path, files=[final_path], report=report)
        finally:
            if os.path.exists(ref_wav):
                try:
                    os.remove(ref_wav)
                except Exception:
                    pass

def process_free_cloning_unified(
    input_mode,
    custom_text,
    srt_file,
    ref_audio,
    lang_label,
    num_steps,
    speed,
    model_engine
):
    if not ref_audio:
        return None, '⚠️ Por favor, envie uma amostra de áudio com a voz a ser clonada.'

    if input_mode == '📄 Importar Legenda (.SRT)':
        if not srt_file:
            return None, '⚠️ Por favor, envie um arquivo de legenda .SRT válido.'
    else:
        if not custom_text or not custom_text.strip():
            return None, '⚠️ Por favor, digite o texto a ser falado.'

    if not job_manager.is_running():
        ok, msg = job_manager.start_job(
            'free_cloning',
            run_free_cloning_worker,
            input_mode=input_mode,
            custom_text=custom_text,
            srt_file=srt_file,
            ref_audio=ref_audio,
            lang_label=lang_label,
            num_steps=num_steps,
            speed=speed,
            model_engine=model_engine
        )
        if not ok:
            return None, f"⚠️ {msg}"

    while job_manager.is_running():
        time.sleep(0.5)

    st = job_manager.get_state()
    if st['status'] == 'completed':
        return st['preview_audio'], st['status_report']
    elif st['status'] == 'cancelled':
        return None, '⏹️ Geração cancelada pelo usuário.'
    else:
        return None, f"❌ Erro: {st['error_message']}"

def render_workflow_steps(completed=0, current=None):
    labels = ['Entrada', 'Transcrição', 'Idiomas', 'Terminal', 'Resultados']
    items = []
    for number, label in enumerate(labels, start=1):
        state_class = ' completed' if number <= completed else (' current' if number == current else '')
        items.append(f"<div class='matraca-step{state_class}'><span class='step-dot'>{number}</span><span>{label}</span></div>")
    return "<div class='matraca-stepper'>" + ''.join(items) + "</div>"

TRANSCRIBE_PROGRESS_RUNNING = """<div class='transcribe-progress running'><div class='progress-copy'><strong>Transcrevendo e analisando áudio...</strong><span>O Whisper está processando o arquivo. Aguarde a conclusão.</span></div><div class='progress-track'><span></span></div></div>"""
TRANSCRIBE_PROGRESS_DONE = """<div class='transcribe-progress done'><div class='progress-copy'><strong>Transcrição concluída</strong><span>O texto está pronto para revisão.</span></div><div class='progress-track'><span></span></div></div>"""
TRANSCRIBE_PROGRESS_ERROR = """<div class='transcribe-progress error'><div class='progress-copy'><strong>Não foi possível concluir a transcrição</strong><span>Confira a mensagem acima e tente novamente.</span></div><div class='progress-track'><span></span></div></div>"""

def finish_transcription_ui(audio_file, transcription, selected_languages):
    success = bool((transcription or '').strip())
    completed = 3 if success and selected_languages else (2 if success else (1 if audio_file else 0))
    progress_html = TRANSCRIBE_PROGRESS_DONE if success else TRANSCRIBE_PROGRESS_ERROR
    return gr.update(value=progress_html, visible=True), render_workflow_steps(completed)

def update_configuration_steps(audio_file, transcription, selected_languages):
    if not audio_file:
        return render_workflow_steps(0)
    if not (transcription or '').strip():
        return render_workflow_steps(1)
    return render_workflow_steps(3 if selected_languages else 2)

def finish_dubbing_steps(files):
    return render_workflow_steps(5 if files else 3)

# ==============================================================================
# 🔄 SINCRONIZAÇÃO AUTOMÁTICA DA INTERFACE (APÓS F5 / RECONEXÃO)
# ==============================================================================
def sync_ui_job_state():
    """
    Executado periodicamente em segundo plano pelo timer do Gradio (ação padrão do app).
    Recupera imediatamente o estado e logs do JobManager se o usuário der F5 na página.
    """
    st = job_manager.get_state()
    banner = render_job_monitor_banner(st)
    status = st['status']

    if status == 'running':
        status_display = f"### 🟡 Dublagem em Execução em Segundo Plano\n- **ID:** `{st['job_id']}` | ⏱️ **Decorrido:** `{st['elapsed']:.1f}s`\n- **Etapa:** {st['stage']}\n\n> 🛡️ *Este processo continua ativo mesmo se você fechar ou recarregar a página (F5).*"
        steps = render_workflow_steps(3, current=4)
        files_out = st['output_files'] if st['output_files'] else gr.skip()
        trans_out = st['translations_summary'] if st['translations_summary'] else gr.skip()
        return banner, status_display, st['logs_text'], steps, files_out, trans_out, gr.skip()

    elif status == 'completed':
        steps = render_workflow_steps(5)
        return banner, st['status_report'], st['logs_text'], steps, st['output_files'], st['translations_summary'], get_existing_outputs()

    elif status == 'cancelled':
        return banner, "⚠️ **Processo interrompido pelo usuário.** Cache de GPU liberado.", st['logs_text'], render_workflow_steps(3), gr.skip(), gr.skip(), gr.skip()

    elif status == 'error':
        return banner, f"❌ **Falha na execução:** `{st['error_message']}`", st['logs_text'], gr.skip(), gr.skip(), gr.skip(), gr.skip()

    # Ocioso / Sem alterações
    return banner, gr.skip(), gr.skip(), gr.skip(), gr.skip(), gr.skip(), gr.skip()

MATRACA_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');
:root { --matraca-green: #08a66c; --matraca-green-dark: #087a54; --matraca-mint: #eafaf4; --matraca-navy: #13294b; --matraca-blue: #2383e2; --matraca-violet: #7657d5; --matraca-border: #dce5ec; --matraca-danger: #ff0040; --matraca-danger-dark: #d60035; }
footer { display: none !important; }
.gradio-container { max-width: none !important; width: 100% !important; padding: 22px 28px 34px !important; background: #ffffff; }
.gradio-container, .gradio-container button, .gradio-container input, .gradio-container textarea, .gradio-container select { font-family: 'Inter', 'Segoe UI Variable', 'Segoe UI', Roboto, Helvetica, Arial, sans-serif !important; font-feature-settings: 'cv02', 'cv03', 'cv04', 'cv11'; }
.gradio-container textarea { font-size: .95rem !important; line-height: 1.62 !important; letter-spacing: -.006em; }
.gradio-container button { border-radius: 8px !important; }
#matraca-header { margin-bottom: 14px; }
.matraca-header { display: flex; align-items: center; justify-content: space-between; gap: 18px; padding: 2px 4px 14px; border-bottom: 1px solid var(--matraca-border); }
.matraca-brand { display: flex; align-items: center; gap: 13px; }
.matraca-logo { flex: 0 0 auto; font-size: 34px; line-height: 1; }
.matraca-title { margin: 0; color: var(--matraca-navy); font-size: clamp(1.35rem, 2.4vw, 2rem); line-height: 1.1; font-weight: 750; letter-spacing: -.025em; }
.matraca-subtitle { margin: 5px 0 0; color: #64748b; font-size: .94rem; }
#main-tabs > .tab-nav { gap: 6px; border-bottom: 1px solid var(--matraca-border); }
#main-tabs > .tab-nav button { padding: 10px 18px; border-radius: 10px 10px 0 0; font-weight: 750 !important; }
.matraca-stepper { display: grid; grid-template-columns: repeat(5, 1fr); gap: 12px; width: 100%; max-width: 1180px; margin: 14px auto 18px; }
.matraca-step { display: flex; align-items: center; gap: 9px; color: #64748b; font-size: .82rem; font-weight: 650; min-width: 0; transition: color .25s ease; }
.matraca-step:not(:last-child)::after { content: ''; height: 2px; flex: 1; min-width: 12px; background: #dce5ec; border-radius: 2px; }
.step-dot { flex: 0 0 29px; height: 29px; display: grid; place-items: center; border-radius: 50%; background: #e7edf2; color: var(--matraca-navy); font-weight: 800; }
.matraca-step.completed { color: var(--matraca-green-dark); }
.matraca-step.completed .step-dot { color: white; background: var(--matraca-green); box-shadow: 0 5px 12px rgba(8,166,108,.22); }
.matraca-step.completed:not(:last-child)::after { background: #65d3ab; }
.matraca-step.current { color: var(--matraca-blue); }
.matraca-step.current .step-dot { color: var(--matraca-blue); background: #e8f3ff; outline: 2px solid #7db9ef; outline-offset: 2px; }
#dub-workspace, #free-workspace { gap: 16px; }
.matraca-card { background: var(--block-background-fill, #fff) !important; border: 1px solid var(--matraca-border) !important; border-radius: 14px !important; padding: 16px !important; box-shadow: 0 5px 18px rgba(19,41,75,.045); gap: 12px !important; }
.matraca-section-title { display: flex; align-items: flex-start; gap: 11px; margin-bottom: 2px; }
.section-number { flex: 0 0 36px; height: 36px; display: grid; place-items: center; border-radius: 50%; color: #fff; background: var(--matraca-green); font-size: 1rem; font-weight: 800; box-shadow: 0 5px 12px rgba(8,166,108,.18); }
.section-number.blue { background: var(--matraca-blue); box-shadow: 0 5px 12px rgba(35,131,226,.18); }
.section-number.violet { background: var(--matraca-violet); box-shadow: 0 5px 12px rgba(118,87,213,.18); }
.matraca-section-title h2 { margin: 0; color: var(--matraca-navy); font-size: 1.03rem; line-height: 1.25; }
.matraca-section-title p { margin: 3px 0 0; color: #718096; font-size: .81rem; line-height: 1.35; }
.compact-card { padding: 12px !important; background: color-mix(in srgb, var(--block-background-fill, #fff) 96%, #edf5f8) !important; border: 1px solid #e5ebf0 !important; border-radius: 11px !important; }
.phase-stack { gap: 16px !important; }
#dub-workspace { width: 100%; }
#target-languages .wrap { display: flex !important; flex-direction: row !important; flex-wrap: wrap !important; justify-content: center !important; gap: 8px !important; }
#target-languages .wrap > label { flex: 0 1 auto !important; min-width: 205px; margin: 0 !important; }
.language-actions { max-width: 360px; }
.transcribe-progress { padding: 12px 14px; border: 1px solid #cfe0ec; border-radius: 11px; background: #f7fbfe; }
.progress-copy { display: flex; justify-content: space-between; gap: 12px; margin-bottom: 9px; color: #476078; font-size: .78rem; }
.progress-copy strong { color: var(--matraca-navy); font-size: .84rem; }
.progress-track { height: 8px; overflow: hidden; border-radius: 999px; background: #e1e9ef; }
.progress-track span { display: block; height: 100%; border-radius: inherit; background: var(--matraca-green); }
.transcribe-progress.running .progress-track span { width: 38%; animation: matraca-progress 1.35s ease-in-out infinite; }
.transcribe-progress.done { border-color: #b9e6d4; background: #f0fbf7; }
.transcribe-progress.done .progress-track span { width: 100%; }
.transcribe-progress.error { border-color: #f2c4c4; background: #fff6f6; }
.transcribe-progress.error .progress-track span { width: 100%; background: #dc5a5a; }
@keyframes matraca-progress { 0% { transform: translateX(-110%); } 55% { transform: translateX(105%); } 100% { transform: translateX(270%); } }
#transcribe-button, #dub-button, #free-button { min-height: 44px; font-weight: 750; border-radius: 10px; }
#transcribe-button { width: min(100%, 560px) !important; flex: none !important; align-self: center !important; margin: 2px auto 4px !important; }
#dub-button, #free-button { background: var(--matraca-green) !important; border-color: var(--matraca-green) !important; }
#dub-button:hover, #free-button:hover { background: var(--matraca-green-dark) !important; border-color: var(--matraca-green-dark) !important; }
#stop-button, #stop-free-button { min-height: 44px; border-radius: 999px; background: var(--button-secondary-background-fill, #fff) !important; color: var(--body-text-color, var(--matraca-navy)) !important; border: 1px solid #d7d7d7 !important; font-weight: 650; transition: background-color .18s ease, border-color .18s ease, color .18s ease, box-shadow .18s ease, transform .12s ease; }
#stop-button:hover, #stop-free-button:hover, #stop-button:focus-visible, #stop-free-button:focus-visible { color: #fff !important; background: var(--matraca-danger) !important; border-color: var(--matraca-danger) !important; box-shadow: 0 6px 16px rgba(255,0,64,.24); }
#stop-button:active, #stop-free-button:active { color: #fff !important; background: var(--matraca-danger-dark) !important; border-color: var(--matraca-danger-dark) !important; box-shadow: 0 3px 8px rgba(214,0,53,.28); transform: translateY(1px); }
#status-card { padding: 12px 15px !important; border: 1px solid #bfe8d7 !important; border-radius: 13px !important; background: var(--matraca-mint) !important; min-height: 72px; }
#status-card h3, #status-card p { margin-top: 0; margin-bottom: 5px; color: #176647; }
#console-card textarea { font-family: ui-monospace, SFMono-Regular, Consolas, monospace !important; font-size: .78rem !important; line-height: 1.55 !important; }
#translations-output textarea { height: 320px !important; min-height: 240px !important; max-height: 420px !important; overflow-y: auto !important; resize: vertical !important; scrollbar-gutter: stable; scrollbar-width: thin; scrollbar-color: #93a4b6 #e7edf2; }
#translations-output textarea::-webkit-scrollbar { width: 10px; }
#translations-output textarea::-webkit-scrollbar-track { background: #e7edf2; border-radius: 999px; }
#translations-output textarea::-webkit-scrollbar-thumb { background: #93a4b6; border: 2px solid #e7edf2; border-radius: 999px; }
#translations-output textarea::-webkit-scrollbar-thumb:hover { background: #6f8295; }
#results-tabs > .tab-nav { display: grid; grid-template-columns: repeat(3, 1fr); gap: 3px; }
#results-tabs > .tab-nav button { justify-content: center; font-size: .82rem; font-weight: 700; }
.mini-actions button { font-size: .78rem !important; }
.form-row { align-items: stretch; }
.form-row > div { min-width: 0; }
.free-intro { margin: 2px 0 12px; color: #64748b; }
.orig-lang-centered { max-width: 580px !important; margin: 8px auto 0 !important; width: 100% !important; text-align: center !important; }
.orig-lang-centered label { text-align: center !important; }
.model-selection-card { display: flex !important; flex-direction: column !important; align-items: center !important; text-align: center !important; }
.free-model-stack { flex-direction: column !important; align-items: center !important; text-align: center !important; }
.free-model-stack > div { flex: 0 1 auto !important; width: min(100%, 720px) !important; }
.free-stepper { max-width: 860px; }
#model-engine-free, #language-choice-free { width: min(100%, 720px) !important; margin: 0 auto !important; text-align: center !important; }
#model-engine-free > label, #language-choice-free > label { display: block !important; text-align: center !important; }
#model-engine-free .wrap { justify-content: center !important; }
#model-engine-dub { display: flex !important; flex-direction: column !important; align-items: center !important; width: 100% !important; }
#model-engine-dub > label { display: flex !important; flex-direction: column !important; align-items: center !important; text-align: center !important; margin: 0 auto 6px !important; }
#model-engine-dub .wrap { display: flex !important; justify-content: center !important; align-items: center !important; gap: 12px !important; width: 100% !important; }
#model-engine-dub .wrap > label { flex: 0 1 auto !important; text-align: center !important; }
.language-actions-centered { display: flex !important; justify-content: center !important; align-items: center !important; gap: 12px !important; max-width: 380px !important; margin: 10px auto 4px !important; }
#select-all-langs-button { min-height: 36px; background: var(--button-secondary-background-fill, #fff) !important; color: var(--body-text-color, var(--matraca-navy)) !important; border: 1px solid #d7d7d7 !important; font-weight: 650; transition: background-color .18s ease, border-color .18s ease, color .18s ease, box-shadow .18s ease, transform .12s ease; }
#select-all-langs-button:hover, #select-all-langs-button:focus-visible { background: #f1f5f9 !important; border-color: #b8c4cf !important; box-shadow: 0 3px 10px rgba(19,41,75,.10); }
#clear-langs-button { min-height: 36px; border-radius: 999px; background: var(--button-secondary-background-fill, #fff) !important; color: var(--body-text-color, var(--matraca-navy)) !important; border: 1px solid #d7d7d7 !important; font-weight: 650; transition: background-color .18s ease, border-color .18s ease, color .18s ease, box-shadow .18s ease, transform .12s ease; }
#clear-langs-button:hover, #clear-langs-button:focus-visible { color: #fff !important; background: var(--matraca-danger) !important; border-color: var(--matraca-danger) !important; box-shadow: 0 6px 16px rgba(255,0,64,.24); }
#clear-langs-button:active { color: #fff !important; background: var(--matraca-danger-dark) !important; border-color: var(--matraca-danger-dark) !important; box-shadow: 0 3px 8px rgba(214,0,53,.28); transform: translateY(1px); }
.matraca-monitor-card { padding: 12px 16px; border-radius: 12px; margin-bottom: 12px; display: flex; flex-direction: column; gap: 6px; border: 1px solid var(--matraca-border); background: #ffffff; }
.matraca-monitor-card.running { background: #fffdf0; border-color: #f6e05e; }
.matraca-monitor-card.completed { background: #f0fdf4; border-color: #86efac; }
.matraca-monitor-card.cancelled { background: #fef2f2; border-color: #fca5a5; }
.monitor-status-badge { display: inline-flex; align-items: center; gap: 6px; font-size: 0.82rem; font-weight: 750; border-radius: 999px; padding: 3px 12px; width: fit-content; }
.monitor-status-badge.idle { background: #e6fffa; color: #047857; }
.monitor-status-badge.running { background: #fef3c7; color: #b45309; }
.monitor-status-badge.completed { background: #dcfce7; color: #15803d; }
.monitor-status-badge.cancelled { background: #fee2e2; color: #b91c1c; }
.monitor-info { font-size: 0.84rem; color: #334155; }
.monitor-note { font-size: 0.76rem; color: #64748b; }
@media (max-width: 980px) { .gradio-container { padding: 14px !important; } #free-workspace { flex-direction: column; } .matraca-stepper { overflow-x: auto; grid-template-columns: repeat(5, minmax(130px, 1fr)); padding-bottom: 6px; } .matraca-header { align-items: flex-start; } .progress-copy { flex-direction: column; gap: 3px; } }
@media (max-width: 640px) { .matraca-header { flex-direction: column; } #results-tabs > .tab-nav { grid-template-columns: repeat(3, 1fr); } }
"""

with gr.Blocks(title='Matraca Studio - Dublador & Clonador de Voz com IA', theme=gr.themes.Soft(primary_hue='emerald', secondary_hue='blue'), css=MATRACA_CSS) as demo:
    # Sentinela no navegador do cliente: reproduz áudio silencioso e envia ping a cada 25s
    gr.HTML("""
    <audio id="matraca-keepalive-audio" loop autoplay muted src="data:audio/wav;base64,UklGRigAAABXQVZFZm10IBIAAAABAAEARKwAAIhYAQACABAAAABkYXRhAgAAAAEA"></audio>
    <script>
    (function() {
        if (window.__matraca_browser_sentry_active) return;
        window.__matraca_browser_sentry_active = true;
        console.log("🛡️ [Matraca UI] Sentinela do navegador iniciado (áudio silencioso + ping 25s).");
        setInterval(function() {
            try {
                fetch(window.location.href, { method: 'HEAD', mode: 'no-cors' }).catch(function() {});
            } catch(e) {}
        }, 25000);
        function unlockAudio() {
            var a = document.getElementById('matraca-keepalive-audio');
            if (a && a.paused) {
                a.play().catch(function() {});
            }
        }
        document.addEventListener('pointerdown', unlockAudio, { once: true });
        document.addEventListener('keydown', unlockAudio, { once: true });
    })();
    </script>
    """)

    gr.HTML("""
    <div id='matraca-header' class='matraca-header'>
        <div class='matraca-brand'>
            <div class='matraca-logo'>🎙️</div>
            <div><h1 class='matraca-title'>Matraca Studio</h1><p class='matraca-subtitle'>Dublagem e clonagem de voz com IA</p></div>
        </div>
    </div>
    """)

    with gr.Tabs(elem_id='main-tabs'):
        # --- ABA 1: DUBLAGEM SINCRONIZADA DE ÁUDIO ---
        with gr.TabItem('🎙️ Dublagem sincronizada'):
            workflow_steps = gr.HTML(render_workflow_steps(0), elem_id='workflow-stepper')
            with gr.Column(elem_id='dub-workspace', elem_classes=['phase-stack']):
                with gr.Column(elem_classes=['matraca-card']):
                    gr.HTML("""<div class='matraca-section-title'><span class='section-number'>1</span><div><h2>Áudio de entrada</h2><p>Envie seu áudio ou grave diretamente no microfone.</p></div></div>""")
                    input_audio = gr.Audio(
                        label='Arquivo de áudio (WAV, MP3, M4A, OGG)',
                        type='filepath'
                    )
                    with gr.Column(elem_classes=['orig-lang-centered']):
                        orig_audio_lang = gr.Dropdown(
                            choices=list(ORIG_LANG_CHOICES.keys()),
                            value='🇧🇷 Português (pt-BR)',
                            label='🌐 Idioma original',
                            info='Ajuda o Whisper a transcrever no idioma correto.'
                        )
                        with gr.Accordion('🎙️ Referência vocal opcional', open=False):
                            custom_ref_voice = gr.Audio(
                                label='5 a 10 segundos de voz limpa',
                                type='filepath'
                            )

                with gr.Column(elem_classes=['matraca-card']):
                    gr.HTML("""<div class='matraca-section-title'><span class='section-number blue'>2</span><div><h2>Transcrição</h2><p>Transcreva o áudio e revise o texto antes da dublagem.</p></div></div>""")
                    btn_transcribe = gr.Button('🔍 Transcrever e analisar', variant='primary', size='md', elem_id='transcribe-button')
                    status_transcribe = gr.Markdown('')
                    transcribe_progress = gr.HTML('', visible=False, elem_id='transcribe-progress')
                    txt_orig = gr.Textbox(
                        label='Transcrição do áudio original',
                        placeholder='A transcrição aparecerá aqui. Você também pode digitar ou colar o texto...',
                        lines=6,
                        interactive=True
                    )

                with gr.Column(elem_classes=['matraca-card']):
                    gr.HTML("""<div class='matraca-section-title'><span class='section-number violet'>3</span><div><h2>Configuração da dublagem</h2><p>Escolha o modelo, os idiomas e as opções de sincronização.</p></div></div>""")
                    with gr.Column(elem_classes=['compact-card', 'model-selection-card']):
                        model_engine_dub = gr.Radio(
                            choices=[
                                '⚡ Qwen3-TTS 1.7B (Alibaba - Alta Fidelidade & Expressividade)',
                                '🎙️ OmniVoice (k2-fsa - Rápido e Leve)'
                            ],
                            value='⚡ Qwen3-TTS 1.7B (Alibaba - Alta Fidelidade & Expressividade)',
                            label='🧠 Modelo de IA',
                            info='Qwen3-TTS: 9 idiomas. OmniVoice: 10 idiomas, incluindo Árabe.',
                            elem_id='model-engine-dub'
                        )
                    with gr.Column(elem_classes=['compact-card']):
                        target_langs = gr.CheckboxGroup(
                            choices=QWEN_SUPPORTED_LABELS,
                            value=['🇺🇸 Inglês (English)', '🇪🇸 Espanhol (Español)'],
                            label='🌐 Idiomas de destino',
                            info='Selecione um ou vários idiomas.',
                            elem_id='target-languages'
                        )
                        with gr.Row(elem_classes=['mini-actions', 'language-actions-centered']):
                            btn_select_all = gr.Button('Selecionar todos', size='sm', elem_id='select-all-langs-button')
                            btn_clear_all = gr.Button('Limpar', variant='stop', size='sm', elem_id='clear-langs-button')
                    with gr.Column(elem_classes=['compact-card']):
                        sync_checkbox = gr.Checkbox(
                            value=True,
                            label='⏱️ Sincronizar linha do tempo e duração',
                            info='Preserva introdução e encerramento e ajusta o tempo da fala.'
                        )
                        with gr.Accordion('⚙️ Configurações avançadas', open=False, visible=False):
                            steps_slider = gr.Slider(minimum=16, maximum=64, value=32, step=4, label='Passos de Difusão (OmniVoice - 32 recomendado)')
                            speed_slider = gr.Slider(minimum=0.8, maximum=1.3, value=1.0, step=0.05, label='Velocidade Base da Fala')

                with gr.Column(elem_classes=['matraca-card']):
                    gr.HTML("""<div class='matraca-section-title'><span class='section-number'>4</span><div><h2>Terminal e monitor de execução</h2><p>Acompanhe em tempo real cada etapa da dublagem (protegido contra F5 e desconexões).</p></div></div>""")

                    job_monitor_banner = gr.HTML(render_job_monitor_banner(), elem_id='job-monitor-banner')
                    status_label = gr.Markdown('### 🟢 Pronto para processar\nEnvie um áudio, revise a transcrição e escolha os idiomas.', elem_id='status-card')
                    with gr.Row():
                        btn_dub = gr.Button('▶ Dublar e sincronizar', variant='primary', size='lg', scale=3, elem_id='dub-button')
                        btn_stop = gr.Button('■ Cancelar', variant='stop', size='lg', scale=1, elem_id='stop-button')
                    console_logs = gr.Textbox(
                        label='📟 Log em tempo real',
                        placeholder='O progresso detalhado de cada bloco e idioma aparecerá aqui...',
                        lines=10,
                        max_lines=18,
                        interactive=False,
                        autoscroll=True,
                        elem_id='console-card'
                    )

                with gr.Column(elem_classes=['matraca-card']):
                    gr.HTML("""<div class='matraca-section-title'><span class='section-number'>5</span><div><h2>Resultados</h2><p>Baixe e consulte os arquivos gerados.</p></div></div>""")
                    with gr.Tabs(elem_id='results-tabs'):
                        with gr.TabItem('⬇ Downloads'):
                            output_files = gr.File(
                                label='Áudios dublados (WAV)',
                                file_count='multiple',
                                type='filepath',
                                interactive=False
                            )
                            zip_download = gr.File(label='Pacote ZIP completo', interactive=False)
                        with gr.TabItem('🕘 Histórico'):
                            gr.Markdown('Arquivos preservados na pasta `./outputs` durante a sessão do Colab.')
                            with gr.Row(elem_classes=['mini-actions']):
                                btn_refresh_history = gr.Button('↻ Atualizar lista', size='sm')
                                btn_download_zip = gr.Button('📦 Gerar ZIP', size='sm')
                            history_files = gr.File(
                                value=get_existing_outputs,
                                label='Arquivos da sessão',
                                file_count='multiple',
                                type='filepath',
                                interactive=False
                            )
                        with gr.TabItem('📝 Traduções'):
                            txt_trans = gr.Textbox(
                                label='Traduções geradas por idioma',
                                lines=12,
                                max_lines=12,
                                interactive=False,
                                autoscroll=True,
                                elem_id='translations-output'
                            )

            # Dinâmica de idiomas ao trocar de modelo:
            model_engine_dub.change(
                fn=on_model_selection_changed,
                inputs=[model_engine_dub, target_langs],
                outputs=[target_langs]
            )

            btn_select_all.click(
                fn=lambda m: QWEN_SUPPORTED_LABELS if 'Qwen' in m else OMNIVOICE_SUPPORTED_LABELS,
                inputs=[model_engine_dub],
                outputs=[target_langs]
            )
            btn_clear_all.click(fn=lambda: [], outputs=[target_langs])

            input_audio.change(
                fn=lambda audio: render_workflow_steps(1 if audio else 0),
                inputs=[input_audio],
                outputs=[workflow_steps],
                queue=False
            )
            target_langs.change(
                fn=update_configuration_steps,
                inputs=[input_audio, txt_orig, target_langs],
                outputs=[workflow_steps],
                queue=False
            )
            txt_orig.change(
                fn=update_configuration_steps,
                inputs=[input_audio, txt_orig, target_langs],
                outputs=[workflow_steps],
                queue=False
            )

            transcribe_start = btn_transcribe.click(
                fn=lambda: gr.update(value=TRANSCRIBE_PROGRESS_RUNNING, visible=True),
                inputs=[],
                outputs=[transcribe_progress],
                queue=False,
                show_progress="hidden"
            )
            transcribe_event = transcribe_start.then(
                fn=transcribe_only,
                inputs=[input_audio, orig_audio_lang],
                outputs=[txt_orig, status_transcribe],
                show_progress="hidden"
            )
            transcribe_event.then(
                fn=finish_transcription_ui,
                inputs=[input_audio, txt_orig, target_langs],
                outputs=[transcribe_progress, workflow_steps],
                queue=False,
                show_progress="hidden"
            )

            dub_start = btn_dub.click(
                fn=lambda: render_workflow_steps(3, current=4),
                inputs=[],
                outputs=[workflow_steps],
                queue=False,
                show_progress="hidden"
            )
            dub_event = dub_start.then(
                fn=process_dubbing,
                inputs=[input_audio, orig_audio_lang, txt_orig, target_langs, custom_ref_voice, sync_checkbox, steps_slider, speed_slider, model_engine_dub],
                outputs=[output_files, status_label, console_logs, txt_trans, job_monitor_banner],
                show_progress="hidden"
            )
            dub_event.then(
                fn=finish_dubbing_steps,
                inputs=[output_files],
                outputs=[workflow_steps],
                queue=False,
                show_progress="hidden"
            )

            btn_stop.click(
                fn=stop_process,
                inputs=[],
                outputs=[console_logs],
                show_progress="hidden"
            )

            btn_refresh_history.click(
                fn=get_existing_outputs,
                outputs=[history_files],
                show_progress="hidden"
            )

            btn_download_zip.click(
                fn=create_zip_outputs,
                outputs=[zip_download],
                show_progress="hidden"
            )



        # --- ABA 2: CLONAGEM LIVRE (TEXTO OU LEGENDA SRT) ---
        with gr.TabItem('🎤 Clonagem livre'):
            gr.HTML("""<div class='matraca-stepper free-stepper' style='grid-template-columns:repeat(3,1fr)'><div class='matraca-step active'><span class='step-dot'>1</span><span>Voz</span></div><div class='matraca-step'><span class='step-dot'>2</span><span>Conteúdo</span></div><div class='matraca-step'><span class='step-dot'>3</span><span>Resultado</span></div></div>""")
            with gr.Column(elem_id='free-workspace'):
                with gr.Column(scale=3, min_width=620):
                    with gr.Column(elem_classes=['matraca-card']):
                        gr.HTML("""<div class='matraca-section-title'><span class='section-number'>1</span><div><h2>Voz de referência</h2><p>Envie uma amostra limpa da voz que será clonada.</p></div></div>""")
                        ref_audio_free = gr.Audio(label='Áudio de referência', type='filepath')
                    with gr.Column(elem_classes=['matraca-card']):
                        gr.HTML("""<div class='matraca-section-title'><span class='section-number blue'>2</span><div><h2>Conteúdo e configuração</h2><p>Digite o texto ou importe uma legenda SRT.</p></div></div>""")
                        with gr.Row(elem_classes=['form-row', 'free-model-stack']):
                            with gr.Column(scale=1):
                                model_engine_free = gr.Radio(
                                    choices=[
                                        '⚡ Qwen3-TTS 1.7B (Alibaba - Alta Fidelidade & Expressividade)',
                                        '🎙️ OmniVoice (k2-fsa - Rápido e Leve)'
                                    ],
                                    value='⚡ Qwen3-TTS 1.7B (Alibaba - Alta Fidelidade & Expressividade)',
                                    label='🧠 Modelo de IA',
                            elem_id='model-engine-free'
                                )
                            with gr.Column(scale=1):
                                lang_choice_free = gr.Dropdown(
                                    choices=OMNIVOICE_SUPPORTED_LABELS,
                                    value='🇧🇷 Português Brasileiro (pt-BR)',
                                    label='🌐 Idioma da fala',
                                    elem_id='language-choice-free'
                                )

                        input_mode = gr.Radio(
                            choices=['✍️ Digitar Texto', '📄 Importar Legenda (.SRT)'],
                            value='✍️ Digitar Texto',
                            label='📝 Método de entrada'
                        )

                        custom_text = gr.Textbox(
                            label='Texto a ser falado',
                            placeholder='Digite o texto que será sintetizado com a voz de referência...',
                            lines=6,
                            visible=True
                        )
                        srt_file = gr.File(
                            label='Legenda SRT com marcações de tempo',
                            file_types=['.srt'],
                            type='filepath',
                            visible=False
                        )

                        with gr.Accordion('⚙️ Configurações avançadas', open=False, visible=False):
                            with gr.Row():
                                steps_free = gr.Slider(minimum=16, maximum=64, value=32, step=4, label='Passos de Difusão (OmniVoice)')
                                speed_free = gr.Slider(minimum=0.7, maximum=1.4, value=1.0, step=0.05, label='Velocidade')

                    def toggle_input_mode(choice):
                        if choice == '📄 Importar Legenda (.SRT)':
                            return gr.update(visible=False), gr.update(visible=True)
                        return gr.update(visible=True), gr.update(visible=False)

                    input_mode.change(fn=toggle_input_mode, inputs=[input_mode], outputs=[custom_text, srt_file])

                with gr.Column(scale=2, min_width=480):
                    with gr.Column(elem_classes=['matraca-card']):
                        gr.HTML("""<div class='matraca-section-title'><span class='section-number violet'>3</span><div><h2>Gerar e ouvir</h2><p>Acompanhe o status e faça a prévia do áudio sintetizado.</p></div></div>""")
                        with gr.Row():
                            btn_free = gr.Button('▶ Gerar áudio com minha voz', variant='primary', size='lg', scale=3, elem_id='free-button')
                            btn_stop_free = gr.Button('■ Cancelar', variant='stop', size='lg', scale=1, elem_id='stop-free-button')
                        status_free = gr.Markdown('')
                        audio_free_out = gr.Audio(label='🔊 Áudio sintetizado', type='filepath', interactive=False)

            free_event = btn_free.click(
                fn=process_free_cloning_unified,
                inputs=[input_mode, custom_text, srt_file, ref_audio_free, lang_choice_free, steps_free, speed_free, model_engine_free],
                outputs=[audio_free_out, status_free],
                show_progress="hidden"
            )

            btn_stop_free.click(
                fn=stop_process,
                inputs=[],
                outputs=[status_free],
                show_progress="hidden"
            )

    # Sentinela de sincronização automática com gr.Timer (a cada 2 segundos - ação padrão do app)
    if hasattr(gr, 'Timer'):
        auto_timer = gr.Timer(value=2.0, active=True)
        auto_timer.tick(
            fn=sync_ui_job_state,
            inputs=[],
            outputs=[job_monitor_banner, status_label, console_logs, workflow_steps, output_files, txt_trans, history_files],
            queue=False,
            show_progress="hidden"
        )

# Fila estritamente sequencial (default_concurrency_limit=1): impede qualquer execução simultânea
demo.queue(max_size=32, default_concurrency_limit=1).launch(
    share=True,
    debug=True,
    allowed_paths=[OUTPUTS_DIR]
)


In [ ]:
# @title Passo 5: Baixar Todos os Áudios Dublados (.ZIP)
# @markdown Execute esta célula a qualquer momento para compactar e baixar todos os arquivos de áudio gerados na sessão (.WAV) diretamente para seu computador.

import os
import glob
import shutil
try:
    from google.colab import files
    in_colab = True
except ImportError:
    in_colab = False

OUTPUTS_DIR = os.path.abspath("./outputs")
zip_output = "/content/audios_dublados"
zip_file = f"{zip_output}.zip"

if os.path.exists(zip_file):
    try:
        os.remove(zip_file)
    except Exception:
        pass

wav_files = sorted(glob.glob(os.path.join(OUTPUTS_DIR, "*.wav")), key=os.path.getmtime, reverse=True)

if not wav_files:
    print("⚠️ Nenhum arquivo de áudio WAV encontrado na pasta ./outputs!")
else:
    print(f"📦 Compactando {len(wav_files)} áudio(s) gerado(s):")
    for f in wav_files:
        size_kb = os.path.getsize(f) / 1024
        print(f"  🎵 {os.path.basename(f)} ({size_kb:.1f} KB)")
    
    shutil.make_archive(zip_output, 'zip', OUTPUTS_DIR)
    total_kb = os.path.getsize(zip_file) / 1024
    print(f"\n🚀 Pacote compactado com sucesso: {os.path.basename(zip_file)} ({total_kb:.1f} KB)!")
    if in_colab:
        files.download(zip_file)
    else:
        print(f"Arquivo salvo localmente em: {zip_file}")
